# LeetCode 296 — Best Meeting Point (Hard) bằng Python

> **Lưu ý: đây là bài Premium** (bị khóa trên LeetCode, chỉ tài khoản trả phí mới mở và nộp được). Notebook tự chứa đầy đủ đề, ví dụ, constraints và bộ test, nên bạn vẫn học và chạy thử trọn vẹn mà không cần tài khoản Premium.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minTotalDistance(self, grid: List[List[int]]) -> int:

```

Bài này mang nhãn **Hard**, nhưng code cuối cùng chỉ khoảng 15 dòng. Cái khó nằm ở **hai nhận xét toán học** mà nếu chưa ai chỉ thì rất khó tự nghĩ ra:

1. Khoảng cách Manhattan **tách được** thành "phần hàng" cộng "phần cột", nên bài 2 chiều biến thành **hai bài 1 chiều**.
2. Trên một đường thẳng, điểm làm tổng khoảng cách nhỏ nhất là **trung vị (median)** — *không phải* trung bình cộng.

Notebook đi từ con số 0: tọa độ trên lưới, khoảng cách "đi bộ trong khu phố", rồi từng bước tới lời giải `O(m·n)`.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc lưới 2 chiều `grid[r][c]` (hàng, cột), duyệt lưới bằng hai vòng `for` lồng nhau và **vẽ lưới bằng ASCII** để tự đếm tay.
2. **Khoảng cách Manhattan** `|r1 - r2| + |c1 - c2|` từ căn bản: vì sao lại là công thức này, khác khoảng cách Euclid (đường chim bay) thế nào.
3. Năm cách giải, đi từ trực giác lên tối ưu:
   - **Cách 1 — Brute force**: thử mọi ô làm điểm hẹn, cộng khoảng cách tới mọi người — `O(m·n·k)`.
   - **Cách 2 — BFS từ mỗi người**: học BFS + `deque`, rồi thấy rằng lưới **không có vật cản** thì BFS là thừa: khoảng cách tính thẳng bằng công thức.
   - **Cách 3 — Tách hàng/cột + sort + median**: tổng 2 chiều = tổng theo hàng + tổng theo cột; mỗi bài 1 chiều có đáp án là **median** — `O(m·n + k log k)`.
   - **Cách 4 (cách chính) — Duyệt đúng thứ tự**: gom hàng khi duyệt theo hàng, gom cột khi duyệt theo cột → hai list **tự sắp xếp sẵn**, bỏ hẳn bước sort — `O(m·n)`.
   - **Cách 5 — Hai con trỏ**: cộng hiệu của cặp người xa nhau nhất từ hai đầu, **không cần biết median**.
4. Hai cách chứng minh median là tối ưu (dịch điểm sang trái/phải; ghép cặp các đoạn lồng nhau) và kiểm chứng bằng thử số ngẫu nhiên.
5. Code nộp LeetCode, test đối chiếu chéo cả 5 cách, lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan và 6 bài tập có đáp án.

Ký hiệu dùng xuyên suốt: `m` = số hàng, `n` = số cột, `k` = số người (số ô có giá trị 1).

## 1. Đề bài (theo LeetCode)

Given an `m x n` binary grid `grid` where each `1` marks the home of one friend, return *the minimal* ***total travel distance***.

The **total travel distance** is the sum of the distances between the houses of the friends and the meeting point.

The distance is calculated using **Manhattan Distance**, where `distance(p1, p2) = |p2.x - p1.x| + |p2.y - p1.y|`.

**Example 1**

```text
Input:  grid = [[1,0,0,0,1],[0,0,0,0,0],[0,0,1,0,0]]
Output: 6
Explanation: Given three friends living at (0,0), (0,4), and (2,2).
The point (0,2) is an ideal meeting point, as the total travel distance of 2 + 2 + 2 = 6 is minimal.
So return 6.
```

**Example 2**

```text
Input:  grid = [[1,1]]
Output: 1
```

**Constraints**

- `m == grid.length`
- `n == grid[i].length`
- `1 <= m, n <= 200`
- `grid[i][j]` is either `0` or `1`.
- There will be **at least two** friends in the grid.

### Dịch nhanh

- `grid` là một lưới `m` hàng × `n` cột; mỗi ô là `0` (ô trống) hoặc `1` (nhà của một người bạn).
- Ta chọn **một ô** làm điểm hẹn (meeting point). Mỗi người đi từ nhà mình tới đó; quãng đường của mỗi người đo bằng **khoảng cách Manhattan**.
- **Tổng quãng đường** = cộng quãng đường của tất cả mọi người. Hãy trả về tổng **nhỏ nhất** có thể.
- Điểm hẹn có thể là ô trống hoặc ô có nhà (ở Example 1, ô `(0,2)` là ô trống). Đề chỉ hỏi **con số** tổng quãng đường, không hỏi vị trí.
- "At least two friends": lưới luôn có ít nhất hai số `1`.

> Gợi ý (hint) LeetCode đưa kèm bài này: *"Try to solve it in one dimension first. How can this solution apply to the two dimension case?"* — tức là "hãy giải bài 1 chiều trước". Notebook sẽ đi đúng con đường đó, nhưng bắt đầu từ cách ngây thơ nhất để thấy **vì sao** cần tới nó.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

grid1 = [[1, 0, 0, 0, 1],
         [0, 0, 0, 0, 0],
         [0, 0, 1, 0, 0]]
grid2 = [[1, 1]]

print("Example 1: m =", len(grid1), "hàng, n =", len(grid1[0]), "cột -> mong đợi 6")
print("Example 2: m =", len(grid2), "hàng, n =", len(grid2[0]), "cột -> mong đợi 1")

# sum(row) cộng các số 1 trong một hàng; cộng tiếp qua mọi hàng = số người
print("Số người ở Example 1:", sum(sum(row) for row in grid1))

Example 1: m = 3 hàng, n = 5 cột -> mong đợi 6
Example 2: m = 1 hàng, n = 2 cột -> mong đợi 1
Số người ở Example 1: 3


## 2. Nền tảng 1 — Lưới 2 chiều và tọa độ `(hàng, cột)`

### 2.1. `grid[r][c]` đọc thế nào?

`grid` là **list của các list**: mỗi phần tử của `grid` là một **hàng** (row), mỗi hàng là một list các ô.

- `grid[r]` → cả hàng thứ `r` (đếm từ 0, đi **từ trên xuống**).
- `grid[r][c]` → ô ở hàng `r`, cột `c` (cột đếm từ 0, đi **từ trái sang phải**).
- `m = len(grid)` = số hàng; `n = len(grid[0])` = số cột (độ dài của một hàng).

Với Example 1:

```text
         c=0 c=1 c=2 c=3 c=4
  r=0  [  1,  0,  0,  0,  1 ]
  r=1  [  0,  0,  0,  0,  0 ]
  r=2  [  0,  0,  1,  0,  0 ]
```

Ba người bạn ở `(0,0)`, `(0,4)`, `(2,2)` — đúng như phần Explanation của đề. Đề viết `p.x`, `p.y`, còn ta dùng `(r, c)`: cứ hiểu "x" là chỉ số hàng, "y" là chỉ số cột. Công thức Manhattan cộng hai phần đối xứng nên gọi tên thế nào cũng không đổi kết quả.

Để tìm tất cả các nhà, ta dùng **hai vòng `for` lồng nhau**: vòng ngoài chạy qua từng hàng `r`, vòng trong chạy qua từng cột `c` của hàng đó. Thứ tự duyệt "hết hàng này mới sang hàng kia" gọi là **row-major** (duyệt theo hàng). Hãy nhớ tên này — nó sẽ là chìa khóa của cách chính ở mục 9.

In [2]:
print("grid1[0]    =", grid1[0], "<- cả hàng 0")
print("grid1[0][4] =", grid1[0][4], "<- hàng 0, cột 4: có nhà")
print("grid1[1][2] =", grid1[1][2], "<- hàng 1, cột 2: ô trống")

homes = []
for r in range(len(grid1)):          # vòng ngoài: từng hàng, từ trên xuống
    for c in range(len(grid1[0])):   # vòng trong: từng cột, từ trái sang phải
        if grid1[r][c] == 1:
            homes.append((r, c))     # lưu tọa độ dạng tuple (r, c)
print("Các nhà theo thứ tự duyệt row-major:", homes)

grid1[0]    = [1, 0, 0, 0, 1] <- cả hàng 0
grid1[0][4] = 1 <- hàng 0, cột 4: có nhà
grid1[1][2] = 0 <- hàng 1, cột 2: ô trống
Các nhà theo thứ tự duyệt row-major: [(0, 0), (0, 4), (2, 2)]


### 2.2. Vẽ lưới bằng ASCII cho dễ nhìn

Nhìn list lồng nhau rất mỏi mắt. Ta viết một hàm nhỏ `draw_grid` chỉ để **nhìn** (không thuộc lời giải):

- `H` (home) = nhà của một người bạn, `.` = ô trống,
- `M` = điểm hẹn nằm trên ô trống, `@` = điểm hẹn trùng đúng một nhà,
- tham số `marks` (tùy chọn) là một `dict` `{(r, c): ký tự}` để ghi ký tự riêng lên vài ô — ta sẽ dùng nó để vẽ đường đi ở mục 3.

Hàng trên cùng in chỉ số cột, đầu mỗi dòng in chỉ số hàng. `marks or {}` nghĩa là: nếu không truyền `marks` (giá trị `None`) thì dùng `dict` rỗng.

In [3]:
def draw_grid(grid, meet=None, marks=None):
    """In lưới: H = nhà, . = trống, M/@ = điểm hẹn, marks = {(r, c): ký tự}."""
    m, n = len(grid), len(grid[0])
    marks = marks or {}
    print("       " + " ".join(str(c % 10) for c in range(n)))   # tiêu đề cột
    print("     +" + "--" * n)
    for r in range(m):
        cells = []
        for c in range(n):
            if (r, c) == meet:
                ch = "@" if grid[r][c] == 1 else "M"
            elif (r, c) in marks:
                ch = marks[(r, c)]
            elif grid[r][c] == 1:
                ch = "H"
            else:
                ch = "."
            cells.append(ch)
        print(f"r={r:<2} | " + " ".join(cells))

print("Example 1, điểm hẹn (0, 2):")
draw_grid(grid1, meet=(0, 2))
print()
print("Example 2:")
draw_grid(grid2)

Example 1, điểm hẹn (0, 2):
       0 1 2 3 4
     +----------
r=0  | H . M . H
r=1  | . . . . .
r=2  | . . H . .

Example 2:
       0 1
     +----
r=0  | H H


## 3. Nền tảng 2 — Khoảng cách Manhattan từ căn bản

### 3.1. Đi bộ trong khu phố ô bàn cờ

Hãy tưởng tượng khu Manhattan ở New York: đường phố kẻ ô vuông như bàn cờ, bạn **không thể đi xuyên qua tòa nhà** theo đường chéo. Mỗi bước chỉ được đi **lên, xuống, trái hoặc phải** đúng một ô.

Muốn đi từ ô `(r1, c1)` tới ô `(r2, c2)`:

- theo chiều dọc phải đi **ít nhất** `|r1 - r2|` bước (chênh lệch số hàng);
- theo chiều ngang phải đi **ít nhất** `|c1 - c2|` bước (chênh lệch số cột);
- mỗi bước chỉ làm thay đổi **một** trong hai (hàng *hoặc* cột) đúng 1 đơn vị, nên không bước nào "làm hộ" được cả hai việc.

Vì vậy độ dài đường đi ngắn nhất là:

```text
manhattan((r1, c1), (r2, c2)) = |r1 - r2| + |c1 - c2|
```

Ký hiệu `|a|` là **giá trị tuyệt đối** (absolute value): bỏ dấu âm đi. `|3| = 3`, `|-3| = 3`, `|0| = 0`. Trong Python đó là hàm `abs()`. Ta cần trị tuyệt đối vì khoảng cách không bao giờ âm: đi từ hàng 0 tới hàng 2 hay từ hàng 2 về hàng 0 đều là 2 bước.

Ví dụ: từ `(0, 0)` tới `(2, 2)` của Example 1: `|0 - 2| + |0 - 2| = 2 + 2 = 4`.

In [4]:
print("abs(3)  =", abs(3))
print("abs(-3) =", abs(-3))
print("abs(0 - 2) =", abs(0 - 2), "| abs(2 - 0) =", abs(2 - 0), "<- thứ tự trừ không quan trọng")

def manhattan(p, q):
    """Khoảng cách Manhattan giữa hai ô p = (r1, c1) và q = (r2, c2)."""
    return abs(p[0] - q[0]) + abs(p[1] - q[1])

print("manhattan((0,0), (2,2)) =", manhattan((0, 0), (2, 2)))
print("manhattan((0,4), (0,2)) =", manhattan((0, 4), (0, 2)))
print("manhattan((2,2), (2,2)) =", manhattan((2, 2), (2, 2)), "<- đứng tại chỗ")

abs(3)  = 3
abs(-3) = 3
abs(0 - 2) = 2 | abs(2 - 0) = 2 <- thứ tự trừ không quan trọng
manhattan((0,0), (2,2)) = 4
manhattan((0,4), (0,2)) = 2
manhattan((2,2), (2,2)) = 0 <- đứng tại chỗ


### 3.2. Có nhiều đường ngắn nhất — nhưng chúng dài bằng nhau

Đi từ `S = (0, 0)` tới `E = (2, 3)`: cần 2 bước xuống và 3 bước sang phải, tổng `2 + 3 = 5` bước. **Thứ tự** các bước thì tùy ý: đi hết sang phải rồi mới đi xuống (hình chữ L), hay đi "bậc thang" xen kẽ — miễn là **không bước nào đi ngược hướng** về phía xa `E`, độ dài luôn là 5. Trong hình bên dưới, các chữ số là thứ tự từng bước.

So với **khoảng cách Euclid** (đường chim bay, kéo thước thẳng): `sqrt(2² + 3²) ≈ 3.61`. Euclid ngắn hơn vì được cắt chéo — nhưng đề bài nói rõ dùng Manhattan, nên ta **không** dùng căn bậc hai. Chính nhờ không có căn bậc hai mà bài này có lời giải đẹp; bản Euclid khó hơn rất nhiều (xem mục 17).

`[[0] * 4 for _ in range(3)]` tạo một lưới 3 × 4 toàn số 0 (mục 5.2 sẽ giải thích vì sao phải viết như vậy).

In [5]:
import math

empty = [[0] * 4 for _ in range(3)]     # lưới 3 x 4 toàn ô trống, chỉ để vẽ

path_L     = {(0, 0): "S", (0, 1): "1", (0, 2): "2", (0, 3): "3", (1, 3): "4", (2, 3): "E"}
path_stair = {(0, 0): "S", (1, 0): "1", (1, 1): "2", (2, 1): "3", (2, 2): "4", (2, 3): "E"}

print("Đường chữ L (bước 5 là tới E):")
draw_grid(empty, marks=path_L)
print("\nĐường bậc thang (cũng 5 bước):")
draw_grid(empty, marks=path_stair)

S, E = (0, 0), (2, 3)
print("\nManhattan(S, E) =", manhattan(S, E))
print("Euclid(S, E)    =", round(math.hypot(E[0] - S[0], E[1] - S[1]), 2), "(đường chim bay, KHÔNG dùng ở bài này)")

Đường chữ L (bước 5 là tới E):
       0 1 2 3
     +--------
r=0  | S 1 2 3
r=1  | . . . 4
r=2  | . . . E

Đường bậc thang (cũng 5 bước):
       0 1 2 3
     +--------
r=0  | S . . .
r=1  | 1 2 . .
r=2  | . 3 4 E

Manhattan(S, E) = 5
Euclid(S, E)    = 3.61 (đường chim bay, KHÔNG dùng ở bài này)


### 3.3. Tổng quãng đường khi chọn một điểm hẹn

Với điểm hẹn `P = (R, C)`, tổng quãng đường là:

```text
total(R, C) = Σ ( |r_i - R| + |c_i - C| )      (cộng qua mọi người bạn i)
```

Ký hiệu `Σ` (sigma) chỉ có nghĩa "cộng tất cả lại". Trong Python có hai cách viết, kết quả như nhau:

```python
# cách 1: vòng for + biến cộng dồn
total = 0
for r, c in homes:
    total += abs(r - R) + abs(c - C)

# cách 2: sum() + generator expression — y hệt cách 1, chỉ viết gọn trên một dòng
total = sum(abs(r - R) + abs(c - C) for r, c in homes)
```

`for r, c in homes` là **unpacking** (tách tuple): mỗi phần tử `(0, 4)` của `homes` được tách thành `r = 0`, `c = 4`.

**Đếm tay Example 1** (nhà ở `(0,0)`, `(0,4)`, `(2,2)`):

| Điểm hẹn | tới (0,0) | tới (0,4) | tới (2,2) | Tổng |
|---|---:|---:|---:|---:|
| `(0,2)` (ô trống) | 2 | 2 | 2 | **6** |
| `(1,2)` | 3 | 3 | 1 | 7 |
| `(0,1)` | 1 | 3 | 3 | 7 |
| `(0,0)` (nhà) | 0 | 4 | 4 | 8 |
| `(2,2)` (nhà) | 4 | 4 | 0 | 8 |
| `(1,1)` | 2 | 4 | 2 | 8 |

Để ý: điểm tốt nhất `(0,2)` **không phải nhà của ai** — hẹn ở nhà bất kỳ người nào cũng tốn 8. Ghi nhớ điều này, nó gắn với một lỗi rất hay gặp (mục 15, Lỗi 3).

In [6]:
def total_distance(homes, R, C):
    """Tổng khoảng cách Manhattan từ mọi nhà tới điểm hẹn (R, C)."""
    return sum(abs(r - R) + abs(c - C) for r, c in homes)

homes1 = [(0, 0), (0, 4), (2, 2)]
for P in [(0, 2), (1, 2), (0, 1), (0, 0), (2, 2), (1, 1)]:
    parts = [manhattan(h, P) for h in homes1]
    # *P tách tuple (R, C) thành hai đối số riêng: total_distance(homes1, R, C)
    print(f"điểm hẹn {P}: từng người {parts} -> tổng {total_distance(homes1, *P)}")

điểm hẹn (0, 2): từng người [2, 2, 2] -> tổng 6
điểm hẹn (1, 2): từng người [3, 3, 1] -> tổng 7
điểm hẹn (0, 1): từng người [1, 3, 3] -> tổng 7
điểm hẹn (0, 0): từng người [0, 4, 4] -> tổng 8
điểm hẹn (2, 2): từng người [4, 4, 0] -> tổng 8
điểm hẹn (1, 1): từng người [2, 4, 2] -> tổng 8


## 4. Cách 1 — Brute force: thử mọi ô làm điểm hẹn

Cách tự nhiên nhất mà người mới nào cũng nghĩ ra:

1. Gom tọa độ tất cả các nhà vào list `homes` (có `k` người).
2. Với **mỗi ô** `(R, C)` của lưới (có `m·n` ô), tính `total_distance(homes, R, C)`.
3. Lấy giá trị nhỏ nhất.

Tại sao chỉ cần thử các ô **trong lưới**? Nếu điểm hẹn nằm ngoài lưới, chẳng hạn phía trên hàng 0, thì kéo nó xuống hàng 0 làm **mọi** người đi ít hơn (vì ai cũng ở hàng `≥ 0`). Mục 7 sẽ chứng minh một điều mạnh hơn: luôn có một điểm hẹn tối ưu mà hàng của nó là hàng của một người nào đó và cột của nó là cột của một người nào đó.

`float("inf")` là "dương vô cực": số nào cũng nhỏ hơn nó, nên dùng làm giá trị khởi đầu khi đi tìm min.

In [7]:
def get_homes(grid):
    """Danh sách tọa độ (r, c) của mọi ô bằng 1, theo thứ tự row-major."""
    homes = []
    for r in range(len(grid)):
        for c in range(len(grid[0])):
            if grid[r][c] == 1:
                homes.append((r, c))
    return homes


def min_total_bruteforce(grid):
    m, n = len(grid), len(grid[0])
    homes = get_homes(grid)
    best = float("inf")
    for R in range(m):              # thử mọi hàng làm hàng điểm hẹn
        for C in range(n):          # thử mọi cột làm cột điểm hẹn
            best = min(best, total_distance(homes, R, C))
    return best

print("Example 1:", min_total_bruteforce(grid1), "(mong đợi 6)")
print("Example 2:", min_total_bruteforce(grid2), "(mong đợi 1)")

Example 1: 6 (mong đợi 6)
Example 2: 1 (mong đợi 1)


### 4.1. "Bản đồ chi phí" — nhìn tận mắt mọi lựa chọn

Brute force có một lợi ích sư phạm: ta in được **tổng quãng đường của từng ô** thành một bảng (gọi là *cost map*). Ô có số nhỏ nhất là điểm hẹn tốt nhất.

Quan sát cost map của Example 1 bên dưới, bạn sẽ thấy một điều thú vị: **các hàng của bảng chỉ lệch nhau một hằng số** (hàng 1 = hàng 0 cộng 1 ở mọi cột, hàng 2 = hàng 0 cộng 2). Tương tự, các cột cũng chỉ lệch nhau một hằng số. Đó là dấu hiệu đầu tiên của tính chất "tách được" ở mục 6 — cứ để ý trước đã.

### 4.2. Brute force vướng ở đâu?

Có `m·n` ô, mỗi ô tính tổng qua `k` người → **Time `O(m·n·k)`**, Space `O(k)` cho list `homes`.

Trường hợp xấu nhất theo constraints: lưới 200 × 200 toàn số 1 → `m·n = 40 000` ô và `k = 40 000` người → khoảng `1.6 · 10^9` lần tính khoảng cách. Python làm được cỡ `10^7` phép tính đơn giản mỗi giây → mất **vài phút**, chắc chắn bị TLE (Time Limit Exceeded — chạy quá thời gian cho phép).

In [8]:
def print_cost_map(grid):
    homes = get_homes(grid)
    m, n = len(grid), len(grid[0])
    for R in range(m):
        print(f"R={R}: ", " ".join(f"{total_distance(homes, R, C):>3}" for C in range(n)))

print("Cost map Example 1 (giá trị ở hàng R, cột C = tổng quãng đường nếu hẹn tại (R, C)):")
print_cost_map(grid1)

print("\nƯớc lượng số lần tính khoảng cách của brute force với lưới toàn số 1:")
for size in [10, 50, 100, 200]:
    cells = size * size
    print(f"  {size:>3} x {size:<3}: {cells:>6} ô x {cells:>6} người = {cells * cells:>14,} lần")

Cost map Example 1 (giá trị ở hàng R, cột C = tổng quãng đường nếu hẹn tại (R, C)):
R=0:    8   7   6   7   8
R=1:    9   8   7   8   9
R=2:   10   9   8   9  10

Ước lượng số lần tính khoảng cách của brute force với lưới toàn số 1:
   10 x 10 :    100 ô x    100 người =         10,000 lần
   50 x 50 :   2500 ô x   2500 người =      6,250,000 lần
  100 x 100:  10000 ô x  10000 người =    100,000,000 lần
  200 x 200:  40000 ô x  40000 người =  1,600,000,000 lần


In [9]:
import time
import random

def timed(f, *args):
    """Chạy f(*args), trả về (kết quả, số giây). perf_counter là đồng hồ chính xác cao."""
    t0 = time.perf_counter()
    result = f(*args)
    return result, time.perf_counter() - t0

random.seed(0)
for size in [20, 40]:
    g = [[random.randint(0, 1) for _ in range(size)] for _ in range(size)]
    res, sec = timed(min_total_bruteforce, g)
    print(f"brute force {size}x{size} (~{size * size // 2} người): kết quả {res}, mất {sec:.3f} giây")
print("Cạnh gấp đôi -> số ô gấp 4 và số người gấp ~4 -> thời gian gấp ~16 lần.")
print("Từ 40x40 lên 200x200: cạnh gấp 5 -> thời gian gấp ~5^4 = 625 lần.")

brute force 20x20 (~200 người): kết quả 1973, mất 0.005 giây
brute force 40x40 (~800 người): kết quả 16146, mất 0.122 giây
Cạnh gấp đôi -> số ô gấp 4 và số người gấp ~4 -> thời gian gấp ~16 lần.
Từ 40x40 lên 200x200: cạnh gấp 5 -> thời gian gấp ~5^4 = 625 lần.


## 5. Cách 2 — BFS từ mỗi người (và vì sao không cần)

Nhiều bạn thấy "lưới" + "khoảng cách" là nghĩ ngay tới **BFS** (Breadth-First Search — tìm kiếm theo chiều rộng). Đây là phản xạ tốt cho rất nhiều bài lưới, nên ta học BFS đàng hoàng, rồi xem nó dạy ta điều gì ở bài này.

### 5.1. Hàng đợi `deque` — công cụ của BFS

BFS cần một **hàng đợi** (queue): ai vào trước thì ra trước (FIFO — First In, First Out), giống xếp hàng mua vé. Python có `collections.deque` (double-ended queue — hàng đợi hai đầu):

- `q.append(x)` → thêm `x` vào **cuối** hàng.
- `q.popleft()` → lấy phần tử ở **đầu** hàng ra, tốn `O(1)`.

Dùng `list` thì `lst.pop(0)` cũng lấy được phần tử đầu, nhưng tốn `O(độ dài list)` vì Python phải dời mọi phần tử còn lại lên một chỗ — hàng đợi dài sẽ rất chậm. Vì vậy BFS luôn dùng `deque`.

In [10]:
from collections import deque

q = deque()
q.append("An")
q.append("Bình")
q.append("Chi")
print("Hàng đợi:", list(q))
first = q.popleft()
print("Phục vụ:", first, "-> còn lại:", list(q))
q.append("Dũng")
print("Dũng xếp vào cuối:", list(q))

Hàng đợi: ['An', 'Bình', 'Chi']
Phục vụ: An -> còn lại: ['Bình', 'Chi']
Dũng xếp vào cuối: ['Bình', 'Chi', 'Dũng']


### 5.2. BFS trên lưới: lan ra như vết dầu loang

Ý tưởng: nhỏ một giọt dầu ở ô xuất phát. Sau 1 giây dầu loang sang 4 ô kề (trên, dưới, trái, phải); sau 2 giây loang tiếp... Thời điểm một ô **bị dầu chạm tới lần đầu** chính là khoảng cách ngắn nhất (số bước đi 4 hướng) từ ô xuất phát tới nó.

Cài đặt:

1. `dist` = bảng `m × n`, ban đầu toàn `-1` (nghĩa là "chưa tới"). Gán `dist` của ô xuất phát bằng 0 và cho ô đó vào hàng đợi.
2. Lặp: lấy ô `(r, c)` ở đầu hàng; xét 4 ô kề `(r + dr, c + dc)` với `(dr, dc)` lần lượt là `(-1,0)`, `(1,0)`, `(0,-1)`, `(0,1)`.
3. Ô kề nào **nằm trong lưới** (`0 <= nr < m` và `0 <= nc < n`) và **chưa tới** (`dist == -1`) thì gán `dist = dist[r][c] + 1` rồi cho vào cuối hàng.

Vì hàng đợi là FIFO, mọi ô cách 1 bước được xử lý xong trước mọi ô cách 2 bước, v.v. — nên lần đầu chạm tới luôn là đường ngắn nhất.

**Bẫy cú pháp khi tạo bảng 2 chiều**: phải viết `[[-1] * n for _ in range(m)]` (mỗi vòng tạo một list **mới** cho mỗi hàng). Nếu viết `[[-1] * n] * m` thì `m` hàng là **cùng một list**, sửa một ô sẽ thấy cả cột thay đổi theo! Ô sau có minh họa ở dòng cuối.

`enumerate(table)` cho ta cả chỉ số hàng `R` lẫn nội dung hàng `row` trong cùng một vòng lặp.

In [11]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải

def bfs_from(m, n, start):
    """Bảng số bước (đi 4 hướng) từ start tới mọi ô của lưới m x n không có vật cản."""
    dist = [[-1] * n for _ in range(m)]
    sr, sc = start
    dist[sr][sc] = 0
    q = deque([start])
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return dist

def print_table(table):
    for R, row in enumerate(table):
        print(f"R={R}: ", " ".join(f"{v:>2}" for v in row))

print("BFS từ nhà (0, 0) trên lưới 3 x 5:")
print_table(bfs_from(3, 5, (0, 0)))
print("\nBFS từ nhà (2, 2):")
print_table(bfs_from(3, 5, (2, 2)))

bad = [[-1] * 3] * 2
bad[0][0] = 99
print("\nBẫy [[-1] * 3] * 2 sau khi chỉ sửa bad[0][0]:", bad)

BFS từ nhà (0, 0) trên lưới 3 x 5:
R=0:   0  1  2  3  4
R=1:   1  2  3  4  5
R=2:   2  3  4  5  6

BFS từ nhà (2, 2):
R=0:   4  3  2  3  4
R=1:   3  2  1  2  3
R=2:   2  1  0  1  2

Bẫy [[-1] * 3] * 2 sau khi chỉ sửa bad[0][0]: [[99, -1, -1], [99, -1, -1]]


### 5.3. Cách 2 hoàn chỉnh: cộng bảng BFS của từng người

Chạy BFS từ **mỗi** nhà, được `k` bảng khoảng cách; cộng chúng lại ô-theo-ô thành bảng tổng; lấy ô nhỏ nhất. Kết quả chắc chắn đúng.

Nhưng hãy so bảng BFS ở trên với công thức Manhattan: từ `(0,0)`, ô `(2,3)` có `dist = 5 = |2-0| + |3-0|`; từ `(2,2)`, ô `(0,0)` có `dist = 4 = |0-2| + |0-2|`. Ô code sau, ngoài việc chạy Cách 2, còn kiểm tra **mọi** điểm xuất phát và **mọi** ô của lưới 4 × 6: BFS có lúc nào khác công thức không?

In [12]:
def min_total_bfs(grid):
    m, n = len(grid), len(grid[0])
    total = [[0] * n for _ in range(m)]
    for home in get_homes(grid):
        dist = bfs_from(m, n, home)          # O(m*n) cho mỗi người
        for R in range(m):
            for C in range(n):
                total[R][C] += dist[R][C]
    return min(min(row) for row in total)

print("Example 1:", min_total_bfs(grid1), "(mong đợi 6)")
print("Example 2:", min_total_bfs(grid2), "(mong đợi 1)")

m, n = 4, 6
always_equal = True
for sr in range(m):
    for sc in range(n):
        dist = bfs_from(m, n, (sr, sc))
        for R in range(m):
            for C in range(n):
                if dist[R][C] != manhattan((sr, sc), (R, C)):
                    always_equal = False
print("Lưới 4 x 6 không vật cản: BFS == Manhattan ở mọi ô, mọi điểm xuất phát?", always_equal)

Example 1: 6 (mong đợi 6)
Example 2: 1 (mong đợi 1)
Lưới 4 x 6 không vật cản: BFS == Manhattan ở mọi ô, mọi điểm xuất phát? True


### 5.4. Cách 2 vướng ở đâu — và bài học rút ra

- **Thời gian**: `k` lần BFS, mỗi lần `O(m·n)` → `O(k·m·n)`, **y hệt brute force** (thực tế còn chậm hơn vì có `deque`, bảng `dist`). **Bộ nhớ** `O(m·n)`.
- **Bài học quan trọng**: BFS chỉ thật sự cần khi đường đi bị **cản** (tường, chướng ngại vật, ô cấm đi). Ở bài này lưới **không có vật cản** — đi xuyên qua ô có nhà cũng được — nên khoảng cách BFS **luôn bằng** công thức `|dr| + |dc|`, tính trong `O(1)`. Bài "anh em" có vật cản là LeetCode 317 *Shortest Distance from All Buildings*; ở đó BFS là bắt buộc.

Cả hai cách đều tốn `O(k·m·n)` vì cùng một lý do: **với mỗi điểm hẹn, ta cộng lại `k` khoảng cách từ đầu**. Muốn nhanh hơn, phải hiểu **cấu trúc** của tổng `Σ (|r_i - R| + |c_i - C|)`. Đó là bước đột phá ở mục sau.

## 6. Bước đột phá 1 — Tổng Manhattan **tách được** thành hàng + cột

Viết lại tổng bằng cách **gom các số hạng cùng loại**:

```text
total(R, C) = Σ ( |r_i - R| + |c_i - C| )
            = Σ |r_i - R|   +   Σ |c_i - C|
            =  rowCost(R)   +    colCost(C)
```

- `rowCost(R)` chỉ phụ thuộc vào **hàng** của điểm hẹn `R` và **các hàng** `r_i` của mọi người. Cột chẳng liên quan gì.
- `colCost(C)` chỉ phụ thuộc vào **cột** `C` và **các cột** `c_i`.

Hai phần **độc lập**: chọn `R` không làm thay đổi `colCost`, chọn `C` không làm thay đổi `rowCost`. Vì vậy để tổng nhỏ nhất, ta chọn `R` làm `rowCost` nhỏ nhất **và**, riêng rẽ, chọn `C` làm `colCost` nhỏ nhất:

```text
min total = min_R rowCost(R)  +  min_C colCost(C)
```

(Đó cũng là lý do các hàng của cost map ở mục 4.1 chỉ lệch nhau một hằng số: đi từ hàng `R` sang hàng `R'` thì chỉ `rowCost` thay đổi, `colCost(C)` giữ nguyên ở mọi cột.)

Ví dụ số với Example 1: hàng của 3 người là `[0, 0, 2]`, cột là `[0, 4, 2]`.

- `rowCost(0) = 0 + 0 + 2 = 2`, `rowCost(1) = 1 + 1 + 1 = 3`, `rowCost(2) = 2 + 2 + 0 = 4` → nhỏ nhất là **2** tại `R = 0`.
- `colCost(0), ..., colCost(4) = 6, 5, 4, 5, 6` → nhỏ nhất là **4** tại `C = 2`.
- Đáp án = `2 + 4 = 6`, tại ô `(0, 2)`. Khớp đề!

Bài 2 chiều đã biến thành **hai bài 1 chiều giống hệt nhau**: *"cho một list số nguyên (các vị trí trên một đường thẳng), chọn điểm `x` để `Σ |a_i - x|` nhỏ nhất"*. Ô sau dựng lại toàn bộ cost map chỉ bằng phép cộng `rowCost[R] + colCost[C]` để bạn tự tin vào nhận xét này.

`[r for r, c in homes1]` là **list comprehension**: một vòng `for` viết gọn để tạo list — "với mỗi `(r, c)` trong `homes1`, lấy `r`".

In [13]:
def cost_1d(points, x):
    """Bài 1 chiều: tổng khoảng cách Σ |p - x|."""
    return sum(abs(p - x) for p in points)

rows1 = [r for r, c in homes1]     # hàng của từng người
cols1 = [c for r, c in homes1]     # cột của từng người
row_cost = [cost_1d(rows1, R) for R in range(3)]
col_cost = [cost_1d(cols1, C) for C in range(5)]
print("rows =", rows1, "-> rowCost theo R = 0..2:", row_cost)
print("cols =", cols1, "-> colCost theo C = 0..4:", col_cost)

print("\nCost map dựng bằng rowCost[R] + colCost[C]:")
for R in range(3):
    print(f"R={R}: ", " ".join(f"{row_cost[R] + col_cost[C]:>3}" for C in range(5)))

ok = all(row_cost[R] + col_cost[C] == total_distance(homes1, R, C)
         for R in range(3) for C in range(5))
print("Khớp cost map của brute force ở mọi ô?", ok)

rows = [0, 0, 2] -> rowCost theo R = 0..2: [2, 3, 4]
cols = [0, 4, 2] -> colCost theo C = 0..4: [6, 5, 4, 5, 6]

Cost map dựng bằng rowCost[R] + colCost[C]:
R=0:    8   7   6   7   8
R=1:    9   8   7   8   9
R=2:   10   9   8   9  10
Khớp cost map của brute force ở mọi ô? True


## 7. Bước đột phá 2 — Bài toán 1 chiều: điểm tốt nhất là **trung vị (median)**

### 7.1. Bài toán con phố

Năm người sống dọc một con phố thẳng, ở các vị trí `[1, 2, 7, 9, 10]`. Chọn vị trí `x` để tổng quãng đường `Σ |a_i - x|` nhỏ nhất.

```text
vị trí:  0  1  2  3  4  5  6  7  8  9 10 11
         .  P  P  .  .  .  .  P  .  P  P  .
```

Trực giác đầu tiên của rất nhiều người: chọn **trung bình cộng** (mean) `(1 + 2 + 7 + 9 + 10) / 5 = 5.8`, làm tròn thành 6. Đừng tin trực giác vội — ta thử hết mọi `x` từ 0 tới 11.

Ở dòng cuối, `min((cost, x) for x in ...)` tìm cặp nhỏ nhất: Python so sánh tuple theo **phần tử đầu tiên trước** (cost), nên cặp nhỏ nhất là cặp có cost nhỏ nhất.

In [14]:
xs = [1, 2, 7, 9, 10]
print("x    :", " ".join(f"{x:>3}" for x in range(12)))
print("cost :", " ".join(f"{cost_1d(xs, x):>3}" for x in range(12)))
print("trung bình =", sum(xs) / len(xs), "-> làm tròn 6, cost(6) =", cost_1d(xs, 6))
best_cost, best_x = min((cost_1d(xs, x), x) for x in range(12))
print("nhỏ nhất   =", best_cost, "tại x =", best_x)

x    :   0   1   2   3   4   5   6   7   8   9  10  11
cost :  29  24  21  20  19  18  17  16  17  18  21  26
trung bình = 5.8 -> làm tròn 6, cost(6) = 17
nhỏ nhất   = 16 tại x = 7


Kết quả: `cost(6) = 17` nhưng `cost(7) = 16` — **trung bình không phải đáp án**. Điểm tốt nhất là `x = 7`, đúng bằng **người đứng giữa** khi xếp các vị trí theo thứ tự: `1, 2, [7], 9, 10`. Người đứng giữa đó gọi là **trung vị (median)**.

Nhìn dòng `cost` từ trái sang: giảm dần tới 16 rồi tăng dần — đồ thị hình chữ V (hơi lệch). Vì sao đáy lại nằm đúng ở median?

### 7.2. Chứng minh trực quan: dịch điểm hẹn sang phải một bước

Giả sử điểm hẹn đang ở `x` và ta dịch nó sang `x + 1`:

- Mỗi người ở **bên trái hoặc ngay tại** `x` (vị trí `≤ x`) phải đi **thêm 1** bước.
- Mỗi người ở **bên phải** `x` (vị trí `> x`, tức `≥ x + 1`) được đi **bớt 1** bước.

Gọi `L` = số người có vị trí `≤ x`, `R` = số người có vị trí `> x`. Khi đó:

```text
cost(x + 1) - cost(x) = L - R
```

- Nếu `L < R` (bên phải đông hơn) → dịch sang phải làm tổng **giảm** → nên đi tiếp.
- Nếu `L > R` (bên trái đông hơn) → dịch sang phải làm tổng **tăng** → dừng.
- Nếu `L = R` → tổng **không đổi** (chỉ xảy ra khi số người chẵn, xem 7.3).

Khi `x` tăng dần, `L` chỉ có thể tăng và `R` chỉ có thể giảm, nên `L - R` đổi dấu từ âm sang dương **đúng một lần** — tại chỗ `x` "vượt qua người đứng giữa". Chỗ đó là median. Hàm `trace_shift` dưới đây in `L`, `R` và mức thay đổi tại mỗi `x` với `xs = [1, 2, 7, 9, 10]`, đồng thời `assert` rằng công thức `L - R` luôn khớp với hiệu thật.

In [15]:
def trace_shift(points, lo, hi):
    print(" x | L(<=x) R(>x) | cost(x) | cost(x+1)-cost(x) = L-R")
    for x in range(lo, hi + 1):
        L = sum(1 for p in points if p <= x)
        R = sum(1 for p in points if p > x)
        delta = cost_1d(points, x + 1) - cost_1d(points, x)
        assert delta == L - R                      # công thức dịch điểm luôn đúng
        if delta < 0:
            note = "sang phải có lợi"
        elif delta == 0:
            note = "ngang bằng"
        else:
            note = "sang phải bị thiệt"
        print(f"{x:>2} |   {L}      {R}   | {cost_1d(points, x):>6}  | {delta:>+3}  {note}")

trace_shift(xs, 0, 10)

 x | L(<=x) R(>x) | cost(x) | cost(x+1)-cost(x) = L-R
 0 |   0      5   |     29  |  -5  sang phải có lợi
 1 |   1      4   |     24  |  -3  sang phải có lợi
 2 |   2      3   |     21  |  -1  sang phải có lợi
 3 |   2      3   |     20  |  -1  sang phải có lợi
 4 |   2      3   |     19  |  -1  sang phải có lợi
 5 |   2      3   |     18  |  -1  sang phải có lợi
 6 |   2      3   |     17  |  -1  sang phải có lợi
 7 |   3      2   |     16  |  +1  sang phải bị thiệt
 8 |   3      2   |     17  |  +1  sang phải bị thiệt
 9 |   4      1   |     18  |  +3  sang phải bị thiệt
10 |   5      0   |     21  |  +5  sang phải bị thiệt


Đọc bảng: tới `x = 6` thì `L = 2 < R = 3`, dịch sang 7 giảm 1; tại `x = 7` thì `L = 3 > R = 2`, dịch tiếp sang 8 tăng 1 → **`x = 7` là đáy**. Mọi điểm bên trái 7 đều "có lợi khi đi sang phải", mọi điểm từ 7 trở đi đều "bị thiệt khi đi tiếp sang phải", nên không điểm nào tốt hơn 7. Kết luận với số người **lẻ**: median là điểm tối ưu **duy nhất**.

Lập luận này cũng cho thấy điểm tối ưu luôn nằm trong đoạn `[min, max]` của các vị trí và luôn có một điểm tối ưu trùng vị trí của một người (median) — đúng như mục 4 đã hứa.

### 7.3. Số người chẵn: cả một "đáy phẳng"

Với `xs = [1, 2, 7, 9]` (4 người), hai người đứng giữa là `2` và `7`. Với mọi `x` mà `2 ≤ x < 7` ta có `L = 2 = R`, nên dịch sang phải không làm thay đổi tổng: mọi `x` trong đoạn `[2, 7]` đều cho cùng một tổng nhỏ nhất. Với `k` người (list đã sắp xếp) ta có thể lấy `points[k // 2]` (người giữa bên phải) hoặc `points[(k - 1) // 2]` (người giữa bên trái) — đều đúng. Khi `k` lẻ, hai chỉ số này trùng nhau.

In [16]:
xs_even = [1, 2, 7, 9]
print("x    :", " ".join(f"{x:>3}" for x in range(11)))
print("cost :", " ".join(f"{cost_1d(xs_even, x):>3}" for x in range(11)))
k = len(xs_even)
print("points[k // 2]       =", xs_even[k // 2], "-> cost", cost_1d(xs_even, xs_even[k // 2]))
print("points[(k - 1) // 2] =", xs_even[(k - 1) // 2], "-> cost", cost_1d(xs_even, xs_even[(k - 1) // 2]))

x    :   0   1   2   3   4   5   6   7   8   9  10
cost :  19  15  13  13  13  13  13  13  15  17  21
points[k // 2]       = 7 -> cost 13
points[(k - 1) // 2] = 2 -> cost 13


### 7.4. Vì sao trung bình (mean) lại sai?

Trung bình cộng bị **kéo lệch** bởi người ở rất xa. Ví dụ `[0, 0, 0, 100]`: mean = 25, nên 3 người ở vị trí 0 mỗi người phải đi 25 bước chỉ vì một người ở xa — tổng `75 + 75 = 150`. Chọn median = 0 thì chỉ một người phải đi 100 bước, tổng `100 < 150`.

Theo lý lẽ dịch điểm: đứng ở `x = 25`, dịch sang trái một bước làm 3 người bớt 1 bước và 1 người thêm 1 bước → tổng giảm 2. Cứ thế dịch về 0. Điều ta cần là **cân bằng số người** ở hai bên (median), chứ không phải cân bằng "khối lượng" hai bên (mean). (Mean là đáp án của một bài khác: làm nhỏ nhất tổng **bình phương** khoảng cách `Σ (a_i - x)²`.)

### 7.5. Kiểm chứng bằng thử số

Lập luận đã có, nhưng ta vẫn nên để máy **thử ngẫu nhiên** hàng nghìn list: với mỗi list, tìm cost nhỏ nhất bằng cách thử mọi số nguyên `x` trong một khoảng **rộng hơn** `[min, max]`, rồi so với cost tại cả hai kiểu median. `random.seed(296)` cố định chuỗi số ngẫu nhiên để lần chạy nào cũng ra cùng kết quả.

In [17]:
print("[0, 0, 0, 100]: cost(mean = 25) =", cost_1d([0, 0, 0, 100], 25),
      "| cost(median = 0) =", cost_1d([0, 0, 0, 100], 0))

random.seed(296)
for _ in range(2000):
    k = random.randint(1, 9)
    pts = sorted(random.randint(0, 20) for _ in range(k))
    best = min(cost_1d(pts, x) for x in range(-5, 26))      # thử cả ngoài đoạn [0, 20]
    assert cost_1d(pts, pts[k // 2]) == best                 # median bên phải
    assert cost_1d(pts, pts[(k - 1) // 2]) == best           # median bên trái
print("2000 list ngẫu nhiên: cost(median) luôn bằng cost nhỏ nhất -> PASS")

[0, 0, 0, 100]: cost(mean = 25) = 150 | cost(median = 0) = 100


2000 list ngẫu nhiên: cost(median) luôn bằng cost nhỏ nhất -> PASS


## 8. Cách 3 — Tách hàng/cột + `sorted` + median

Ghép hai bước đột phá lại thành thuật toán:

1. Gom `rows` = hàng của mọi người, `cols` = cột của mọi người.
2. Sắp xếp (sort) hai list.
3. Lấy median mỗi list: `R* = rows[k // 2]`, `C* = cols[k // 2]`.
4. Đáp án = `Σ |r_i - R*| + Σ |c_i - C*|`.

**Python cần biết:**

- `sorted(lst)` trả về **list mới** đã sắp xếp tăng dần (list cũ giữ nguyên); còn `lst.sort()` sắp xếp **tại chỗ** và trả về `None`. Đừng bao giờ viết `lst = lst.sort()`!
- Phần tử giữa của list độ dài `k` (đã sắp xếp) là `lst[k // 2]`; `//` là phép chia lấy phần nguyên: `3 // 2 = 1`, `4 // 2 = 2`.
- `sorted(r for r, c in homes)` nhận thẳng một generator expression, không cần tạo list trung gian.

Đề chỉ hỏi con số tổng, nhưng nhân tiện, ô `(R*, C*)` chính là **một** điểm hẹn tối ưu.

In [18]:
cols_raw = [0, 4, 2]
print("sorted([0, 4, 2]) =", sorted(cols_raw), "| list cũ vẫn là", cols_raw)
print("cols_raw.sort() trả về:", cols_raw.sort(), "| nhưng list đã bị đổi thành", cols_raw)
for lst in [[0, 2, 4], [1, 2, 7, 9]]:
    print(f"list {lst}: k = {len(lst)}, k // 2 = {len(lst) // 2}, phần tử giữa = {lst[len(lst) // 2]}")


def min_total_sort_median(grid):
    homes = get_homes(grid)
    rows = sorted(r for r, c in homes)
    cols = sorted(c for r, c in homes)
    k = len(homes)
    R, C = rows[k // 2], cols[k // 2]
    return sum(abs(r - R) for r in rows) + sum(abs(c - C) for c in cols)

print("\nExample 1:", min_total_sort_median(grid1), "(mong đợi 6)")
print("Example 2:", min_total_sort_median(grid2), "(mong đợi 1)")

sorted([0, 4, 2]) = [0, 2, 4] | list cũ vẫn là [0, 4, 2]
cols_raw.sort() trả về: None | nhưng list đã bị đổi thành [0, 2, 4]
list [0, 2, 4]: k = 3, k // 2 = 1, phần tử giữa = 2
list [1, 2, 7, 9]: k = 4, k // 2 = 2, phần tử giữa = 7

Example 1: 6 (mong đợi 6)
Example 2: 1 (mong đợi 1)


### Cách 3 tốn bao nhiêu? Còn vướng ở đâu?

- Duyệt lưới gom nhà: `O(m·n)`. Sort hai list `k` phần tử: `O(k log k)`. Tính hai tổng: `O(k)`.
- **Time `O(m·n + k log k)`**, **Space `O(k)`**. Với `k ≤ m·n = 40 000` thì chạy trong chớp mắt — đã đủ để Accept.

Từ `O(m·n·k)` xuống gần tuyến tính là một cú nhảy khổng lồ. Nhưng người phỏng vấn thường hỏi tiếp: *"Có bỏ được bước sort không?"* Câu trả lời nằm ngay trong **thứ tự ta duyệt lưới**.

## 9. Cách 4 (cách chính) — Duyệt đúng thứ tự để có sẵn list đã sắp xếp, `O(m·n)`

### 9.1. Quan sát: duyệt row-major thì `rows` tự động tăng dần

Ở `get_homes`, vòng ngoài là `r` chạy `0, 1, 2, ...`. Vậy mỗi khi gặp một nhà, chỉ số hàng ta ghi vào **không bao giờ nhỏ hơn** chỉ số đã ghi trước đó → `rows` **đã được sắp xếp sẵn**, khỏi cần sort!

Nhưng `cols` thì không: Example 1 cho `cols = [0, 4, 2]` (hàng 0 có cột 0 và cột 4, xuống tới hàng 2 mới gặp cột 2).

### 9.2. Mẹo: gom `cols` bằng cách duyệt theo **cột** (column-major)

Đảo hai vòng lặp: vòng ngoài là cột `c`, vòng trong là hàng `r`. Khi đó chỉ số **cột** ghi vào không bao giờ giảm → `cols` cũng sắp xếp sẵn. Ta duyệt lưới **hai lần** (một lần theo hàng, một lần theo cột), mỗi lần `O(m·n)` — tổng vẫn là `O(m·n)`, không còn `log`.

Ô sau in **thứ tự thăm ô** của hai kiểu duyệt trên một lưới 2 × 3 để bạn thấy rõ, rồi gom `rows`, `cols` của Example 1. Cú pháp `[(r, c) for r in range(m) for c in range(n)]` là list comprehension **hai tầng**: đọc từ trái sang phải đúng như hai vòng `for` lồng nhau, vòng viết trước là vòng ngoài.

In [19]:
m, n = 2, 3
row_major = [(r, c) for r in range(m) for c in range(n)]      # r ở vòng ngoài
col_major = [(r, c) for c in range(n) for r in range(m)]      # c ở vòng ngoài
print("row-major   :", row_major)
print("column-major:", col_major)


def collect_rows(grid):
    """Hàng của mọi nhà, duyệt theo hàng -> list tự động tăng dần."""
    rows = []
    for r in range(len(grid)):
        for c in range(len(grid[0])):
            if grid[r][c] == 1:
                rows.append(r)
    return rows


def collect_cols(grid):
    """Cột của mọi nhà, duyệt theo cột -> list tự động tăng dần."""
    cols = []
    for c in range(len(grid[0])):       # vòng ngoài là CỘT
        for r in range(len(grid)):
            if grid[r][c] == 1:
                cols.append(c)
    return cols

print("\nExample 1:")
print("  rows (duyệt theo hàng)              =", collect_rows(grid1))
print("  cols nếu duyệt theo hàng (CHƯA sort) =", [c for r, c in get_homes(grid1)])
print("  cols (duyệt theo cột)               =", collect_cols(grid1))

row-major   : [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]
column-major: [(0, 0), (1, 0), (0, 1), (1, 1), (0, 2), (1, 2)]

Example 1:
  rows (duyệt theo hàng)              = [0, 0, 2]
  cols nếu duyệt theo hàng (CHƯA sort) = [0, 4, 2]
  cols (duyệt theo cột)               = [0, 2, 4]


### 9.3. Ghép lại: lời giải chính

```text
rows = collect_rows(grid)                       # đã tăng dần
cols = collect_cols(grid)                       # đã tăng dần
đáp án = min_distance_1d(rows) + min_distance_1d(cols)
```

Hàm `min_distance_1d(points)` nhận một list **đã sắp xếp**, lấy median `points[len(points) // 2]` rồi cộng các khoảng cách tới nó.

- **Time `O(m·n)`**: hai lần duyệt lưới, mỗi lần `m·n` bước; tính hai tổng `O(k) ≤ O(m·n)`. Không còn `sorted` nào.
- **Space `O(k)`** cho hai list `rows`, `cols`.

In [20]:
def min_distance_1d(points):
    """points ĐÃ SẮP XẾP. Trả về min_x Σ|p - x| bằng cách đặt x tại median."""
    median = points[len(points) // 2]
    return sum(abs(p - median) for p in points)


def min_total_distance(grid):
    rows = collect_rows(grid)
    cols = collect_cols(grid)
    return min_distance_1d(rows) + min_distance_1d(cols)

print("Example 1:", min_total_distance(grid1), "(mong đợi 6)")
print("Example 2:", min_total_distance(grid2), "(mong đợi 1)")

Example 1: 6 (mong đợi 6)
Example 2: 1 (mong đợi 1)


## 10. Cách 5 — Hai con trỏ: tính tổng mà **không cần biết median**

### 10.1. Bổ đề cặp đôi

Lấy hai người ở vị trí `a ≤ b`. Tổng quãng đường của **riêng hai người này** tới điểm `x`:

- Nếu `a ≤ x ≤ b` (`x` nằm giữa họ): `|a - x| + |b - x| = (x - a) + (b - x) = b - a` — **không phụ thuộc `x`**!
- Nếu `x` nằm ngoài đoạn `[a, b]`: tổng **lớn hơn** `b - a` (một người phải đi "quá" người kia rồi mới tới).

Vậy với **mọi** `x`: `|a - x| + |b - x| ≥ b - a`, và dấu bằng xảy ra khi `x` nằm giữa hai người. Ví dụ `a = 2`, `b = 7`:

In [21]:
a, b = 2, 7
for x in range(0, 10):
    s = abs(a - x) + abs(b - x)
    where = "nằm giữa -> đúng bằng b - a" if a <= x <= b else "ở ngoài   -> lớn hơn b - a"
    print(f"x = {x}: |{a}-{x}| + |{b}-{x}| = {s:>2}   ({where})")
print("b - a =", b - a)

x = 0: |2-0| + |7-0| =  9   (ở ngoài   -> lớn hơn b - a)
x = 1: |2-1| + |7-1| =  7   (ở ngoài   -> lớn hơn b - a)
x = 2: |2-2| + |7-2| =  5   (nằm giữa -> đúng bằng b - a)
x = 3: |2-3| + |7-3| =  5   (nằm giữa -> đúng bằng b - a)
x = 4: |2-4| + |7-4| =  5   (nằm giữa -> đúng bằng b - a)
x = 5: |2-5| + |7-5| =  5   (nằm giữa -> đúng bằng b - a)
x = 6: |2-6| + |7-6| =  5   (nằm giữa -> đúng bằng b - a)
x = 7: |2-7| + |7-7| =  5   (nằm giữa -> đúng bằng b - a)
x = 8: |2-8| + |7-8| =  7   (ở ngoài   -> lớn hơn b - a)
x = 9: |2-9| + |7-9| =  9   (ở ngoài   -> lớn hơn b - a)
b - a = 5


### 10.2. Ghép cặp từ hai đầu — các đoạn lồng nhau

Sắp xếp các vị trí: `p[0] ≤ p[1] ≤ ... ≤ p[k-1]`. Ghép người **ngoài cùng bên trái** với người **ngoài cùng bên phải**, rồi cặp kế tiếp tiến vào trong, cứ thế:

```text
xs = [1, 2, 7, 9, 10]

   1  2  7  9  10
   [===========]      cặp (1, 10): đóng góp 10 - 1 = 9
      [=====]         cặp (2, 9):  đóng góp  9 - 2 = 7
         7            người giữa (k lẻ): đóng góp 0
                      tổng = 9 + 7 = 16   (đúng bằng cost(7) ở mục 7)
```

Các đoạn `[1, 10] ⊇ [2, 9] ⊇ [7, 7]` **lồng nhau** như búp bê Nga. Lý lẽ hoàn chỉnh:

1. **Cận dưới**: với *bất kỳ* `x`, mỗi cặp tốn ít nhất `b - a` (bổ đề). Cộng lại: `cost(x) ≥ Σ (b - a)`.
2. **Đạt được**: chọn `x` trong đoạn trong cùng (chính là median) thì `x` nằm giữa **mọi** cặp cùng lúc, nên mọi cặp đều tốn đúng `b - a`.
3. Vậy `min cost = Σ (p[k-1-i] - p[i])` qua các cặp `i < k-1-i`. Người đứng giữa (nếu `k` lẻ) đóng góp 0.

Đây là **chứng minh thứ hai** rằng median là tối ưu, và nó cho ta cách tính bằng **hai con trỏ** (two pointers): con trỏ `i` xuất phát từ đầu trái, `j` từ đầu phải; mỗi bước cộng `points[j] - points[i]` (hiệu của hai người **xa nhau nhất** còn lại) rồi cùng tiến vào giữa; dừng khi `i >= j`. Không cần tính median, không cần `abs`.

Ô sau cài đặt, kèm hàm `trace_two_pointers` in từng cặp cho một ví dụ `k` lẻ và một ví dụ `k` chẵn, rồi ráp vào lưới thành Cách 5.

In [22]:
def min_distance_1d_two_pointers(points):
    """points ĐÃ SẮP XẾP. Cộng hiệu của cặp xa nhau nhất, rồi thu hẹp vào giữa."""
    i, j = 0, len(points) - 1
    total = 0
    while i < j:
        total += points[j] - points[i]
        i += 1
        j -= 1
    return total


def trace_two_pointers(points):
    i, j, total = 0, len(points) - 1, 0
    print(f"  points = {points}")
    while i < j:
        total += points[j] - points[i]
        print(f"    i={i}, j={j}: cặp ({points[i]}, {points[j]}) -> +{points[j] - points[i]}, total = {total}")
        i += 1
        j -= 1
    print(f"    dừng vì i={i} >= j={j}; kết quả {total}")
    return total

trace_two_pointers([1, 2, 7, 9, 10])
trace_two_pointers([1, 2, 7, 9])
print("So với cách median:", min_distance_1d([1, 2, 7, 9, 10]), "và", min_distance_1d([1, 2, 7, 9]))


def min_total_two_pointers(grid):
    return (min_distance_1d_two_pointers(collect_rows(grid))
            + min_distance_1d_two_pointers(collect_cols(grid)))

print("\nExample 1:", min_total_two_pointers(grid1), "(mong đợi 6)")
print("Example 2:", min_total_two_pointers(grid2), "(mong đợi 1)")

  points = [1, 2, 7, 9, 10]
    i=0, j=4: cặp (1, 10) -> +9, total = 9
    i=1, j=3: cặp (2, 9) -> +7, total = 16
    dừng vì i=2 >= j=2; kết quả 16
  points = [1, 2, 7, 9]
    i=0, j=3: cặp (1, 9) -> +8, total = 8
    i=1, j=2: cặp (2, 7) -> +5, total = 13
    dừng vì i=2 >= j=1; kết quả 13
So với cách median: 16 và 13

Example 1: 6 (mong đợi 6)
Example 2: 1 (mong đợi 1)


## 11. Mô phỏng toàn bộ lời giải trên một lưới "khó" hơn

Example 1 hơi dễ (số người lẻ, không ai chung cột). Ta dùng lưới 4 × 6 có **6 người** (số chẵn → có "đáy phẳng"), có **hai người chung cột 4**, hai người chung hàng 0, hai người chung hàng 2, và **cột 3 trống**:

```text
       0 1 2 3 4 5
     +------------
r=0  | H . . . H .
r=1  | . . H . . .
r=2  | . H . . . H
r=3  | . . . . H .
```

| Bước | Cách làm | Kết quả |
|---|---|---|
| Gom hàng | duyệt theo hàng: hàng 0 thêm `[0, 0]`, hàng 1 thêm `[1]`, hàng 2 thêm `[2, 2]`, hàng 3 thêm `[3]` | `rows = [0, 0, 1, 2, 2, 3]` (tự tăng dần) |
| Gom cột | duyệt theo cột: cột 0 → `[0]`, 1 → `[1]`, 2 → `[2]`, 3 → `[]`, 4 → `[4, 4]`, 5 → `[5]` | `cols = [0, 1, 2, 4, 4, 5]` |
| Hai con trỏ trên `rows` | cặp `(0,3)`, `(0,2)`, `(1,2)` | `3 + 2 + 1 = 6` |
| Hai con trỏ trên `cols` | cặp `(0,5)`, `(1,4)`, `(2,4)` | `5 + 3 + 2 = 10` |
| Đáp án | | `6 + 10 = 16` |

Vì `k = 6` chẵn, mọi hàng `R` trong `[rows[2], rows[3]] = [1, 2]` và mọi cột `C` trong `[cols[2], cols[3]] = [2, 4]` đều tối ưu → có **2 × 3 = 6 ô** cùng đạt tổng 16. Cost map ở cuối output đánh dấu `*` các ô đó: một "đáy phẳng" hình chữ nhật — phiên bản 2 chiều của mục 7.3.

Trong code, `rows += found` nghĩa là nối list `found` vào cuối `rows`.

In [23]:
grid_trace = [[1, 0, 0, 0, 1, 0],
              [0, 0, 1, 0, 0, 0],
              [0, 1, 0, 0, 0, 1],
              [0, 0, 0, 0, 1, 0]]

def trace_min_total(grid):
    m, n = len(grid), len(grid[0])
    draw_grid(grid)
    print("[1] Duyệt theo hàng để gom rows:")
    rows = []
    for r in range(m):
        found = [r for c in range(n) if grid[r][c] == 1]
        rows += found
        print(f"    hàng {r}: thêm {found} -> rows = {rows}")
    print("[2] Duyệt theo cột để gom cols:")
    cols = []
    for c in range(n):
        found = [c for r in range(m) if grid[r][c] == 1]
        cols += found
        print(f"    cột {c}: thêm {found} -> cols = {cols}")
    print("[3] Hai con trỏ:")
    row_part = trace_two_pointers(rows)
    col_part = trace_two_pointers(cols)
    total = row_part + col_part
    print(f"[4] Đáp án = {row_part} + {col_part} = {total}")
    return total

ans = trace_min_total(grid_trace)
print("Brute force xác nhận:", min_total_bruteforce(grid_trace))

homes_t = get_homes(grid_trace)
print("Cost map (* = ô tối ưu):")
for R in range(4):
    line = ""
    for C in range(6):
        v = total_distance(homes_t, R, C)
        line += f"{v:>3}{'*' if v == ans else ' '}"
    print(f"  R={R}:{line}")

       0 1 2 3 4 5
     +------------
r=0  | H . . . H .
r=1  | . . H . . .
r=2  | . H . . . H
r=3  | . . . . H .
[1] Duyệt theo hàng để gom rows:
    hàng 0: thêm [0, 0] -> rows = [0, 0]
    hàng 1: thêm [1] -> rows = [0, 0, 1]
    hàng 2: thêm [2, 2] -> rows = [0, 0, 1, 2, 2]
    hàng 3: thêm [3] -> rows = [0, 0, 1, 2, 2, 3]
[2] Duyệt theo cột để gom cols:
    cột 0: thêm [0] -> cols = [0]
    cột 1: thêm [1] -> cols = [0, 1]
    cột 2: thêm [2] -> cols = [0, 1, 2]
    cột 3: thêm [] -> cols = [0, 1, 2]
    cột 4: thêm [4, 4] -> cols = [0, 1, 2, 4, 4]
    cột 5: thêm [5] -> cols = [0, 1, 2, 4, 4, 5]
[3] Hai con trỏ:
  points = [0, 0, 1, 2, 2, 3]
    i=0, j=5: cặp (0, 3) -> +3, total = 3
    i=1, j=4: cặp (0, 2) -> +2, total = 5
    i=2, j=3: cặp (1, 2) -> +1, total = 6
    dừng vì i=3 >= j=2; kết quả 6
  points = [0, 1, 2, 4, 4, 5]
    i=0, j=5: cặp (0, 5) -> +5, total = 5
    i=1, j=4: cặp (1, 4) -> +3, total = 8
    i=2, j=3: cặp (2, 4) -> +2, total = 10
    dừng vì i=3 >= j=2; kết

## 12. Code nộp LeetCode

Bản nộp ghép **Cách 4** (duyệt đúng thứ tự, không sort) với **Cách 5** (hai con trỏ, không cần median, không cần `abs`). Hàm phụ `_min_1d` đặt trong class; tên bắt đầu bằng `_` là quy ước "hàm dùng nội bộ". LeetCode chỉ gọi `minTotalDistance`.

Nếu bạn thích dùng median thì thay thân `_min_1d` bằng hai dòng của `min_distance_1d` (mục 9.3) — vẫn `O(m·n)` và cũng Accept.

In [24]:
from typing import List

class Solution:
    def minTotalDistance(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])

        rows = []                      # duyệt theo hàng -> rows tăng dần
        for r in range(m):
            for c in range(n):
                if grid[r][c] == 1:
                    rows.append(r)

        cols = []                      # duyệt theo cột -> cols tăng dần
        for c in range(n):
            for r in range(m):
                if grid[r][c] == 1:
                    cols.append(c)

        return self._min_1d(rows) + self._min_1d(cols)

    def _min_1d(self, points: List[int]) -> int:
        # points đã sắp xếp: cộng hiệu của cặp xa nhất, thu hẹp dần vào giữa
        i, j = 0, len(points) - 1
        total = 0
        while i < j:
            total += points[j] - points[i]
            i += 1
            j -= 1
        return total

In [25]:
def check(name, grid, expected):
    got = Solution().minTotalDistance(grid)
    assert got == expected, (name, got, expected)
    print(f"{name:<38} -> {got:>4}  PASS")

check("Example 1", [[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]], 6)
check("Example 2", [[1, 1]], 1)
check("1 cột, 2 người kề nhau", [[1], [1]], 1)
check("1 cột, 3 người", [[1, 0], [1, 0], [1, 0]], 2)
check("1 hàng, hai đầu", [[1, 0, 0, 0, 1]], 4)
check("toàn số 1, lưới 3 x 3", [[1] * 3 for _ in range(3)], 12)
check("lưới trace 4 x 6 (k chẵn, chung cột)", grid_trace, 16)

big = [[0] * 200 for _ in range(200)]
big[0][0] = big[199][199] = 1
check("200 x 200, hai góc đối diện", big, 398)
big[0][199] = big[199][0] = 1
check("200 x 200, bốn góc", big, 796)

Example 1                              ->    6  PASS
Example 2                              ->    1  PASS
1 cột, 2 người kề nhau                 ->    1  PASS
1 cột, 3 người                         ->    2  PASS
1 hàng, hai đầu                        ->    4  PASS
toàn số 1, lưới 3 x 3                  ->   12  PASS
lưới trace 4 x 6 (k chẵn, chung cột)   ->   16  PASS
200 x 200, hai góc đối diện            ->  398  PASS
200 x 200, bốn góc                     ->  796  PASS


### 12.1. Test đối chiếu chéo trên lưới ngẫu nhiên

Ví dụ viết tay chỉ phủ được vài tình huống. Ta sinh ngẫu nhiên vài trăm lưới nhỏ (cạnh 1 → 6, mật độ số 1 khác nhau, luôn có ít nhất 2 người như đề đảm bảo) và yêu cầu **cả 6 hàm** — 5 cách đã học cộng class `Solution` — trả cùng một đáp án. Brute force là "trọng tài" đáng tin nhất vì nó thử hết mọi khả năng.

`{name: f(g) for name, f in solvers.items()}` là **dict comprehension**: tạo dict "tên cách → đáp án". `set(...)` bỏ các giá trị trùng; nếu mọi cách cùng đáp án thì `set` chỉ còn **1** phần tử.

### 12.2. Chạy ở giới hạn của đề

Sau đó ta đo thời gian bản nộp trên lưới 200 × 200 (lớn nhất theo constraints) — toàn số 1 và ngẫu nhiên 50%. Với lưới toàn số 1, đáp án tính tay được: mỗi chiều là `200 · Σ|i - 100|` (với `i = 0..199`) `= 200 · 10 000 = 2 000 000`, hai chiều là `4 000 000`.

In [26]:
def random_grid(m, n, density):
    g = [[1 if random.random() < density else 0 for _ in range(n)] for _ in range(m)]
    while sum(sum(row) for row in g) < 2:          # đề đảm bảo có ít nhất 2 người
        g[random.randrange(m)][random.randrange(n)] = 1
    return g

solvers = {
    "bruteforce":   min_total_bruteforce,
    "bfs":          min_total_bfs,
    "sort_median":  min_total_sort_median,
    "scan_median":  min_total_distance,
    "two_pointers": min_total_two_pointers,
    "Solution":     Solution().minTotalDistance,
}

random.seed(296)
tested = 0
while tested < 400:
    m, n = random.randint(1, 6), random.randint(1, 6)
    if m * n < 2:                                   # lưới 1 ô không chứa nổi 2 người
        continue
    g = random_grid(m, n, random.choice([0.1, 0.3, 0.6, 0.9]))
    answers = {name: f(g) for name, f in solvers.items()}
    assert len(set(answers.values())) == 1, (g, answers)
    tested += 1
print(f"{tested} lưới ngẫu nhiên: cả {len(solvers)} hàm cho cùng đáp án -> PASS")

400 lưới ngẫu nhiên: cả 6 hàm cho cùng đáp án -> PASS


In [27]:
random.seed(1)
full = [[1] * 200 for _ in range(200)]
half = [[random.randint(0, 1) for _ in range(200)] for _ in range(200)]

res, sec = timed(Solution().minTotalDistance, full)
assert res == 4_000_000
print(f"Solution, 200x200 toàn số 1 : đáp án {res:,}, mất {sec:.4f} giây")

res, sec = timed(Solution().minTotalDistance, half)
print(f"Solution, 200x200 ngẫu nhiên: đáp án {res:,}, mất {sec:.4f} giây")

res, sec = timed(min_total_sort_median, full)
print(f"Cách 3 (có sort), toàn số 1 : đáp án {res:,}, mất {sec:.4f} giây")

Solution, 200x200 toàn số 1 : đáp án 4,000,000, mất 0.0052 giây
Solution, 200x200 ngẫu nhiên: đáp án 1,989,769, mất 0.0040 giây
Cách 3 (có sort), toàn số 1 : đáp án 4,000,000, mất 0.0117 giây


## 13. Vì sao lời giải đúng? (bốn mắt xích)

1. **Chỉ cần công thức, không cần BFS.** Lưới không có vật cản, nên đường đi ngắn nhất giữa hai ô dài đúng `|dr| + |dc|`: cần ít nhất chừng ấy bước (mỗi bước chỉ đổi hàng *hoặc* cột đi 1), và đi "chữ L" là đạt được (mục 3.1, kiểm chứng bằng BFS ở mục 5.3).
2. **Tách được.** `total(R, C) = rowCost(R) + colCost(C)`, hai phần độc lập → tối ưu riêng từng phần rồi cộng lại là tối ưu chung (mục 6).
3. **Bài 1 chiều.** Với list đã sắp xếp, giá trị nhỏ nhất của `Σ |p - x|` bằng `Σ (p[k-1-i] - p[i])` qua các cặp từ hai đầu, và đạt được tại median. Chứng minh: mỗi cặp tốn `≥ b - a` với **mọi** `x` (cận dưới), còn median nằm giữa **mọi** cặp nên đạt đúng cận dưới (mục 10.2). Lý lẽ dịch điểm ở mục 7.2 cho cùng kết luận từ một góc nhìn khác.
4. **Không cần sort.** **Bất biến** (invariant — điều luôn đúng sau mỗi bước) của vòng duyệt theo hàng: *"`rows` luôn tăng dần, vì chỉ số hàng ở vòng ngoài chỉ tăng"*. Tương tự với `cols` khi vòng ngoài là cột (mục 9). Vậy hai con trỏ / median nhận đúng list đã sắp xếp.

Một câu hỏi tinh tế: nếu điểm hẹn được phép là một điểm **bất kỳ** (tọa độ thực, không cần là ô) thì sao? Cận dưới `Σ (b - a)` ở mắt xích 3 đúng với **mọi** `x` thực, nên đáp án không thể nhỏ hơn; mà median lại là một tọa độ nguyên có thật trong lưới. Vậy đáp án không đổi — đề có hiểu theo kiểu nào cũng đúng.

## 14. Độ phức tạp và so sánh các cách

Ký hiệu: lưới `m × n`, `k` = số người (`2 ≤ k ≤ m·n ≤ 40 000`).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | thử mọi ô, cộng `k` khoảng cách | `O(m·n·k)`, tệ nhất `O((m·n)²)` | `O(k)` | dễ nghĩ nhất; TLE ở 200 × 200 |
| 2. BFS từ mỗi người | `k` lần BFS, cộng các bảng | `O(k·m·n)` | `O(m·n)` | đúng nhưng thừa: không có vật cản |
| 3. Tách + sort + median | hai bài 1 chiều | `O(m·n + k log k)` | `O(k)` | đã Accept, dễ giải thích |
| 4. Duyệt hàng/cột + median | bỏ sort nhờ thứ tự duyệt | `O(m·n)` | `O(k)` | **cách chính** |
| 5. Duyệt hàng/cột + hai con trỏ | cộng cặp xa nhất từ hai đầu | `O(m·n)` | `O(k)` | không cần median, không `abs` — dùng trong bản nộp |

- Không thể nhanh hơn `O(m·n)` về mặt tiệm cận, vì ít nhất phải đọc mọi ô một lần mới biết ai ở đâu.
- Bộ nhớ còn hạ được xuống `O(m + n)` bằng cách chỉ **đếm số người ở mỗi hàng / mỗi cột** thay vì lưu từng người — đó chính là Bài 6 (mini challenge).
- Ở cỡ 200 × 200, Cách 3 và Cách 4/5 đều chạy rất nhanh; khác biệt `log k` gần như không thấy bằng mắt. Giá trị của Cách 4/5 nằm ở **tư duy**: tận dụng thứ tự có sẵn thay vì sort lại.

## 15. Lỗi thường gặp

### Lỗi 1 — Dùng trung bình (mean) thay vì trung vị (median)

```python
# SAI
R = round(sum(rows) / len(rows))
C = round(sum(cols) / len(cols))
```

**Vì sao sai**: mean bị người ở xa kéo lệch (mục 7.4). Phản ví dụ: một hàng có nhà ở cột 0, 1, 7 → mean ≈ 2.67, làm tròn thành 3 → tổng `3 + 2 + 4 = 9`; median là 1 → tổng `1 + 0 + 6 = 7`.

**Sửa**: dùng `points[len(points) // 2]` trên list đã sắp xếp (hoặc hai con trỏ).

### Lỗi 2 — Gom `cols` theo thứ tự duyệt **hàng** rồi lấy phần tử giữa mà quên sort

```python
# SAI
for r in range(m):
    for c in range(n):
        if grid[r][c] == 1:
            rows.append(r)
            cols.append(c)       # cols KHÔNG tăng dần!
C = cols[len(cols) // 2]
```

**Vì sao sai**: `rows` tình cờ đã tăng dần, nhưng `cols` thì không. Example 1: `cols = [0, 4, 2]`, phần tử ở giữa là `4` chứ không phải median `2` → tổng thành `2 + 6 = 8` thay vì 6. Hai con trỏ trên list chưa sắp xếp cũng cho ra con số vô nghĩa.

**Sửa**: duyệt riêng theo cột (Cách 4), hoặc `cols.sort()` trước (Cách 3).

In [28]:
def wrong_mean(grid):
    homes = get_homes(grid)
    rows = [r for r, c in homes]
    cols = [c for r, c in homes]
    R = round(sum(rows) / len(rows))        # SAI: mean
    C = round(sum(cols) / len(cols))
    return total_distance(homes, R, C)

def wrong_unsorted_cols(grid):
    homes = get_homes(grid)
    rows = [r for r, c in homes]
    cols = [c for r, c in homes]            # SAI: chưa sắp xếp
    R = rows[len(rows) // 2]
    C = cols[len(cols) // 2]
    return total_distance(homes, R, C)

g_far = [[1, 1, 0, 0, 0, 0, 0, 1]]
print("Lỗi 1 (mean) trên", g_far, ":", wrong_mean(g_far),
      "| đúng:", Solution().minTotalDistance(g_far))
print("Lỗi 2 (cols chưa sort) trên Example 1:", wrong_unsorted_cols(grid1),
      "| đúng:", Solution().minTotalDistance(grid1))

Lỗi 1 (mean) trên [[1, 1, 0, 0, 0, 0, 0, 1]] : 9 | đúng: 7
Lỗi 2 (cols chưa sort) trên Example 1: 8 | đúng: 6


### Lỗi 3 — Chỉ thử **nhà của bạn bè** làm điểm hẹn

```python
# SAI
return min(total_distance(homes, R, C) for R, C in homes)
```

**Vì sao sai**: trong bài 1 chiều, median luôn là vị trí của một người — nhưng **hàng** median và **cột** median có thể đến từ **hai người khác nhau**, nên ô `(R*, C*)` có thể là ô trống. Example 1: hàng tối ưu là 0 (của người ở `(0,0)` hoặc `(0,4)`), cột tối ưu là 2 (của người ở `(2,2)`) → ô `(0,2)` trống. Nếu chỉ thử 3 nhà thì được 8.

**Sửa**: tách hàng/cột và tối ưu riêng từng chiều.

### Lỗi 4 — Hai con trỏ dừng sai điều kiện: `while i != j`

```python
# SAI
while i != j:
    total += points[j] - points[i]
    i += 1
    j -= 1
```

**Vì sao sai**: với **số người chẵn**, `i` và `j` **nhảy qua nhau** mà không bao giờ bằng nhau (ví dụ `k = 2`: `(i, j) = (0, 1) → (1, 0) → (2, -1) → ...`). Vòng lặp chạy tiếp, cộng thêm các hiệu âm, cuối cùng `points[i]` vượt chỉ số → `IndexError`. Với `k` lẻ thì tình cờ chạy đúng, nên test ít rất dễ không phát hiện.

**Sửa**: `while i < j` (hoặc `while i <= j` cũng đúng, vì khi `i == j` chỉ cộng thêm `0`).

In [29]:
def wrong_only_homes(grid):
    homes = get_homes(grid)
    return min(total_distance(homes, R, C) for R, C in homes)   # SAI: chỉ thử nhà

def wrong_two_pointers(points):
    i, j, total = 0, len(points) - 1, 0
    while i != j:                                               # SAI: phải là i < j
        total += points[j] - points[i]
        i += 1
        j -= 1
    return total

print("Lỗi 3 (chỉ thử nhà) trên Example 1:", wrong_only_homes(grid1),
      "| đúng:", Solution().minTotalDistance(grid1))
print("Lỗi 4 với k lẻ [1, 2, 7]:", wrong_two_pointers([1, 2, 7]),
      "(tình cờ đúng, vì đáp án là 7 - 1 = 6)")
try:
    wrong_two_pointers([0, 1])
except IndexError as e:
    print("Lỗi 4 với k chẵn [0, 1]: IndexError ->", e)

Lỗi 3 (chỉ thử nhà) trên Example 1: 8 | đúng: 6
Lỗi 4 với k lẻ [1, 2, 7]: 6 (tình cờ đúng, vì đáp án là 7 - 1 = 6)
Lỗi 4 với k chẵn [0, 1]: IndexError -> list index out of range


### Lỗi 5 — Nhầm `m` và `n` khi lưới không vuông

`m = len(grid)` là số **hàng**, `n = len(grid[0])` là số **cột**. Khi đảo vòng lặp để duyệt theo cột, rất dễ viết nhầm `for c in range(m)` / `for r in range(n)`. Lưới vuông thì lỗi không lộ ra; Example 2 `[[1, 1]]` (1 hàng × 2 cột) lập tức lộ: hoặc `IndexError`, hoặc bỏ sót người.

**Sửa**: đặt tên rõ ràng `m, n = len(grid), len(grid[0])` ngay đầu hàm, và luôn test ít nhất một lưới **1 hàng** và một lưới **1 cột**.

### Lỗi 6 — Vài lỗi Python lặt vặt

- `rows = rows.sort()` → `rows` thành `None` (vì `.sort()` trả về `None`). Dùng `rows.sort()` hoặc `rows = sorted(rows)`.
- Tạo bảng bằng `[[0] * n] * m` (trong Cách 2) → các hàng dùng chung một list (mục 5.2).
- Dùng `/` thay vì `//` khi lấy chỉ số giữa: `len(points) / 2` là số thực, `points[2.0]` → `TypeError`.
- Tưởng phải trả về **tọa độ** điểm hẹn: đề chỉ hỏi **tổng quãng đường** (một số nguyên).

In [30]:
def wrong_swapped_mn(grid):
    m, n = len(grid), len(grid[0])
    cols = []
    for c in range(m):              # SAI: c phải chạy tới n
        for r in range(n):          # SAI: r phải chạy tới m
            if grid[r][c] == 1:
                cols.append(c)
    return cols

try:
    print(wrong_swapped_mn([[1, 1]]))
except IndexError as e:
    print("Lỗi 5 trên [[1, 1]]: IndexError ->", e)
print("Đúng: collect_cols([[1, 1]]) =", collect_cols([[1, 1]]))

pts = [3, 1, 2]
print("Lỗi 6: pts.sort() trả về", pts.sort(), "(nên rows = rows.sort() sẽ thành None)")
try:
    [0, 1, 2][len(pts) / 2]
except TypeError as e:
    print("Lỗi 6: dùng / làm chỉ số ->", e)

Lỗi 5 trên [[1, 1]]: IndexError -> list index out of range
Đúng: collect_cols([[1, 1]]) = [0, 1]
Lỗi 6: pts.sort() trả về None (nên rows = rows.sort() sẽ thành None)
Lỗi 6: dùng / làm chỉ số -> list indices must be integers or slices, not float


## 16. Mẹo nhận dạng pattern

- **"Tổng khoảng cách Manhattan nhỏ nhất tới một điểm"** → nghĩ ngay: **tách theo từng trục** + **median**. Manhattan là tổng các trị tuyệt đối theo từng trục, nên các trục độc lập với nhau.
- **"Chọn một giá trị `x` để `Σ |a_i - x|` nhỏ nhất"** — làm các số bằng nhau với ít bước `+1/-1` nhất, đặt trạm/bưu điện trên một con đường, chọn thời điểm họp ít lệch nhất... → **median**. Còn `Σ (a_i - x)²` → **mean**.
- **Cần tổng tới median mà không muốn tìm median** → hai con trỏ từ hai đầu, cộng `p[j] - p[i]`.
- **Cần list đã sắp xếp của một loại chỉ số trên lưới** → chọn **thứ tự duyệt** sao cho chỉ số đó nằm ở **vòng ngoài**; tránh được sort.
- **Lưới + khoảng cách**: hỏi ngay *"có vật cản không?"*. Không vật cản → công thức `|dr| + |dc|`. Có vật cản / ô cấm đi → BFS.
- Khi bài 2 chiều có vẻ quá khó, thử xem nó có **tách thành các bài 1 chiều** không — đúng như hint của LeetCode cho bài này.

## 17. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| **462. Minimum Moves to Equal Array Elements II** (Medium) | Đúng là bài 1 chiều: mỗi bước `+1/-1` một phần tử, cần ít bước nhất để mọi phần tử bằng nhau → tổng khoảng cách tới median (Bài 5). |
| **2033. Minimum Operations to Make a Uni-Value Grid** (Medium) | Duỗi lưới thành list, mỗi bước cộng/trừ `x`; kiểm tra mọi số cùng số dư khi chia cho `x`, rồi đưa về median. |
| **2448. Minimum Cost to Make Array Equal** (Hard) | Mỗi phần tử có chi phí riêng cho mỗi bước → **median có trọng số** (weighted median) — họ hàng gần của Bài 6. |
| **1478. Allocate Mailboxes** (Hard) | Đặt `k` hòm thư: chia các nhà thành nhóm liên tiếp (DP), mỗi nhóm đặt hòm thư tại median của nhóm. |
| **317. Shortest Distance from All Buildings** (Hard, Premium) | Giống bài này nhưng có **vật cản** và điểm hẹn phải là ô đất trống → bắt buộc BFS từ mỗi tòa nhà (ở đó Cách 2 mới là đúng hướng). |
| **1515. Best Position for a Service Centre** (Hard) | Khoảng cách **Euclid** → không tách được theo trục, không có công thức median; phải dùng phương pháp lặp (gradient descent, thu hẹp bước nhảy...). |
| **1584. Min Cost to Connect All Points** (Medium) | Cũng dùng khoảng cách Manhattan, nhưng là bài cây khung nhỏ nhất (MST). |

Biến thể tự chế để luyện thêm:

- Mỗi ô chứa **số người** thay vì 0/1 → median có trọng số (Bài 6).
- Điểm hẹn bắt buộc là **ô trống** → công thức `total(R, C) = rowCost(R) + colCost(C)` vẫn đúng, nhưng không còn chọn `R`, `C` độc lập được. Cách làm: tính sẵn `rowCost[R]` cho **mọi** `R` (và `colCost[C]` cho mọi `C`) bằng chính công thức dịch điểm `cost(x + 1) = cost(x) + L - R`, rồi lấy min của `rowCost[R] + colCost[C]` trên các ô trống — vẫn `O(m·n)`.
- Được đi **chéo** (8 hướng) → khoảng cách thành `max(|dr|, |dc|)` (khoảng cách Chebyshev) và **không còn tách trực tiếp** theo hàng/cột; cần thêm mẹo (ví dụ xoay hệ trục 45°).

## 18. Bài tập tự luyện

Làm theo thứ tự, tự làm xong mới xem **Gợi ý** (mục 18.7) và **Đáp án** (mục 19). Các cell TODO chỉ định nghĩa hàm; test để sẵn dưới dạng comment — làm xong thì bỏ dấu `#` để tự kiểm tra.

### 18.1. Bài 1 — Đoán output (không chạy code)

```python
g = [[0, 1, 0],
     [1, 0, 1],
     [0, 1, 0]]
```

a) `collect_rows(g)` trả về gì?
b) `[c for r, c in get_homes(g)]` (cột theo thứ tự duyệt **hàng**) trả về gì? Đã tăng dần chưa?
c) `collect_cols(g)` trả về gì?
d) `Solution().minTotalDistance(g)` bằng bao nhiêu? Điểm hẹn tối ưu là ô nào, ô đó có nhà không?

### 18.2. Bài 2 — Tính tay bài 1 chiều

Năm người sống dọc phố ở các vị trí `[1, 4, 4, 6, 15]` (chú ý: hai người cùng ở vị trí 4).

a) Trung bình cộng bằng bao nhiêu? Tính `cost(mean)`.
b) Median bằng bao nhiêu? Tính `cost(median)`.
c) Lập bảng dịch điểm cho `x = 3, 4, 5`: các cột `L`, `R`, `cost(x)`, `cost(x+1) - cost(x)`. Giải thích vì sao 4 là đáy.
d) Tính lại đáp án bằng hai con trỏ (liệt kê từng cặp).

### 18.3. Bài 3 — Điền TODO: hai con trỏ cho bài 1 chiều

Hoàn thiện hàm nhận list **đã sắp xếp** và trả về `min_x Σ |p - x|` bằng hai con trỏ — không dùng `abs`, không tìm median.

In [31]:
def exercise_min_distance_1d(points):
    """points đã sắp xếp tăng dần. Trả về min_x Σ|p - x| bằng hai con trỏ."""
    # TODO 1: đặt i ở đầu trái, j ở đầu phải, total = 0
    # TODO 2: trong khi i < j: cộng hiệu của cặp xa nhất points[j] - points[i]
    # TODO 3: tiến i sang phải, lùi j sang trái
    # TODO 4: trả về total
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_min_distance_1d([1, 2, 7, 9, 10]) == 16
# assert exercise_min_distance_1d([1, 2, 7, 9]) == 13
# assert exercise_min_distance_1d([5, 5]) == 0
# assert exercise_min_distance_1d([0, 3]) == 3

### 18.4. Bài 4 — Tự viết lại lời giải `O(m·n)`

Không nhìn lại mục 9 và 12, tự viết hàm giải bài 296 trong `O(m·n)`: **không** được dùng `sorted` hay `.sort()`. Được phép gọi lại hàm của Bài 3.

In [32]:
def exercise_min_total_distance(grid):
    """Lời giải O(m*n): gom rows theo hàng, cols theo cột, rồi giải hai bài 1 chiều."""
    # TODO: không dùng sorted / sort
    raise NotImplementedError

# assert exercise_min_total_distance([[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]]) == 6
# assert exercise_min_total_distance([[1, 1]]) == 1
# assert exercise_min_total_distance([[1], [1], [1]]) == 2
# assert exercise_min_total_distance([[1, 0, 0, 0, 1, 0], [0, 0, 1, 0, 0, 0],
#                                     [0, 1, 0, 0, 0, 1], [0, 0, 0, 0, 1, 0]]) == 16

### 18.5. Bài 5 — Biến thể: LeetCode 462. Minimum Moves to Equal Array Elements II

Cho mảng số nguyên `nums` độ dài `n`. Mỗi bước (move) được **tăng hoặc giảm một phần tử đi 1**. Trả về số bước ít nhất để mọi phần tử bằng nhau. (Signature LeetCode: `def minMoves2(self, nums: List[int]) -> int`.)

- `nums = [1, 2, 3]` → `2` (`[1,2,3] → [2,2,3] → [2,2,2]`)
- `nums = [1, 10, 2, 9]` → `16`

Constraints (theo LeetCode): `n == nums.length`, `1 <= nums.length <= 10^5`, `-10^9 <= nums[i] <= 10^9`; test được thiết kế để đáp án vừa số nguyên 32-bit.

Chú ý: mảng **chưa** sắp xếp và có thể có số âm.

In [33]:
def exercise_min_moves2(nums):
    """LeetCode 462: số bước +1/-1 ít nhất để mọi phần tử bằng nhau."""
    # TODO: sắp xếp (bản sao của) nums rồi dùng lại ý tưởng bài 1 chiều
    pass

# assert exercise_min_moves2([1, 2, 3]) == 2
# assert exercise_min_moves2([1, 10, 2, 9]) == 16
# assert exercise_min_moves2([7]) == 0
# assert exercise_min_moves2([-5, 5]) == 10

### 18.6. Bài 6 — Mini challenge: mỗi ô có **nhiều người**, bộ nhớ `O(m + n)`

Biến thể: `grid[r][c]` là số nguyên **≥ 0** = số người sống ở ô đó (một ô có thể có nhiều người). Tổng quãng đường vẫn là tổng khoảng cách Manhattan của **từng người** tới điểm hẹn. Giả sử có ít nhất 1 người.

Yêu cầu: thời gian `O(m·n)`, bộ nhớ phụ `O(m + n)` — **không** được tạo list có một phần tử cho mỗi người (số người có thể rất lớn, ví dụ một ô có `10^6` người).

Ví dụ:

- `[[2, 0, 1]]` → 2 người ở cột 0, 1 người ở cột 2 → hẹn ở cột 0: `0 + 0 + 2 = 2`.
- `[[1, 0, 0], [0, 0, 0], [0, 0, 3]]` → `4` (hẹn ở ô của 3 người, người còn lại đi 4 bước).
- Với lưới chỉ gồm 0/1, đáp án phải trùng `minTotalDistance`.

In [34]:
def exercise_min_total_weighted(grid):
    """grid[r][c] = số người ở ô (r, c). Time O(m*n), bộ nhớ phụ O(m + n)."""
    # TODO 1: row_count[r] = tổng số người ở hàng r; col_count[c] = tổng số người ở cột c
    # TODO 2: viết hàm 1 chiều nhận count[i] = "số người ở vị trí i":
    #         tìm median có trọng số, rồi cộng count[i] * |i - median|
    # TODO 3: đáp án = phần hàng + phần cột
    raise NotImplementedError

# assert exercise_min_total_weighted([[2, 0, 1]]) == 2
# assert exercise_min_total_weighted([[1, 0, 0], [0, 0, 0], [0, 0, 3]]) == 4
# assert exercise_min_total_weighted([[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]]) == 6
# assert exercise_min_total_weighted([[0, 1000000, 0, 1]]) == 2

### 18.7. Gợi ý theo mức

**Bài 1**
1. Viết ra list `homes` theo thứ tự duyệt hàng trước, mọi câu đều suy ra từ đó.
2. Câu c): đảo hai vòng lặp, đi từng cột từ trái sang phải; trong mỗi cột đi từ trên xuống.
3. Câu d): dùng hai con trỏ trên kết quả câu a) và câu c), rồi cộng.

**Bài 2**
1. mean = tổng / số người.
2. Câu c): `L` đếm người có vị trí `≤ x` — nhớ có **2 người** cùng ở 4!
3. `cost(x+1) - cost(x) = L - R`; đáy là chỗ dấu đổi từ âm sang dương.

**Bài 3**
1. Hai đầu tiến dần vào giữa.
2. `i, j = 0, len(points) - 1`, rồi `while i < j:`.
3. Thân vòng: `total += points[j] - points[i]`, sau đó `i += 1` và `j -= 1`.

**Bài 4**
1. Hai lần duyệt lưới, khác nhau ở chỗ vòng nào là vòng ngoài.
2. Khi gom cột: vòng ngoài `for c in range(n)`, vòng trong `for r in range(m)`.
3. Đáp án = `exercise_min_distance_1d(rows) + exercise_min_distance_1d(cols)`.

**Bài 5**
1. Mỗi bước `+1/-1` = đi 1 bước trên trục số; "mọi phần tử bằng `x`" = mọi người cùng tới điểm hẹn `x`.
2. Mảng chưa sắp xếp → `sorted(nums)` (`O(n log n)`, thừa nhanh với `n = 10^5`).
3. Hai con trỏ trên list đã sắp xếp (Bài 3).

**Bài 6**
1. Tách hàng/cột vẫn đúng; với mỗi hàng ta chỉ cần biết **có bao nhiêu người** ở hàng đó.
2. Median có trọng số: duyệt vị trí từ trái sang, cộng dồn số người vào `prefix`; vị trí **đầu tiên** có `2 * prefix >= tổng số người` là một median. (Lý lẽ dịch điểm: tại đó bên trái-hoặc-tại đã chiếm ít nhất một nửa, nên sang phải không có lợi; còn ở vị trí ngay trước nó bên phải đông hơn, nên đi sang phải tới đó là có lợi.)
3. Phần 1 chiều = `Σ count[i] * |i - median|`.

## 19. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

- a) `[0, 1, 1, 2]` — hàng 0 có 1 người, hàng 1 có 2 người, hàng 2 có 1 người.
- b) `[1, 0, 2, 1]` — **chưa** tăng dần (đó là lý do phải duyệt theo cột hoặc sort).
- c) `[0, 1, 1, 2]` — cột 0 có 1 người, cột 1 có 2 người, cột 2 có 1 người.
- d) Hàng: `(2 - 0) + (1 - 1) = 2`; cột: `(2 - 0) + (1 - 1) = 2` → đáp án **4**. Điểm hẹn tối ưu là `(1, 1)` — ô chính giữa, **không có nhà** (bốn người đứng thành hình thoi quanh nó, mỗi người đi 1 bước).

### Đáp án Bài 2

- a) mean = `30 / 5 = 6` → `cost(6) = 5 + 2 + 2 + 0 + 9 = 18`.
- b) median = `4` (phần tử giữa của `[1, 4, 4, 6, 15]`) → `cost(4) = 3 + 0 + 0 + 2 + 11 = 16`.
- c) Bảng dịch điểm:

| x | L (≤ x) | R (> x) | cost(x) | cost(x+1) − cost(x) |
|---:|---:|---:|---:|---:|
| 3 | 1 | 4 | 19 | −3 |
| 4 | 3 | 2 | 16 | +1 |
| 5 | 3 | 2 | 17 | +1 |

Từ 3 sang 4 giảm 3 (bên phải đông hơn), từ 4 sang 5 tăng 1 (bên trái đã đông hơn) → 4 là đáy.

- d) Cặp `(1, 15)` → 14, cặp `(4, 6)` → 2, người giữa `4` → 0. Tổng `14 + 2 = 16`.

Ô sau kiểm chứng Bài 1 và Bài 2 bằng chính các hàm trong notebook.

In [35]:
g_ex1 = [[0, 1, 0],
         [1, 0, 1],
         [0, 1, 0]]
print("Bài 1a:", collect_rows(g_ex1))
print("Bài 1b:", [c for r, c in get_homes(g_ex1)])
print("Bài 1c:", collect_cols(g_ex1))
best_total, best_cell = min((total_distance(get_homes(g_ex1), R, C), (R, C))
                            for R in range(3) for C in range(3))
print("Bài 1d:", Solution().minTotalDistance(g_ex1), "| ô tối ưu:", best_cell,
      "| có nhà?", g_ex1[best_cell[0]][best_cell[1]] == 1)
assert Solution().minTotalDistance(g_ex1) == best_total == 4

xs2 = [1, 4, 4, 6, 15]
print("\nBài 2a: mean =", sum(xs2) / len(xs2), "-> cost(6) =", cost_1d(xs2, 6))
print("Bài 2b: median =", xs2[len(xs2) // 2], "-> cost(4) =", cost_1d(xs2, 4))
print("Bài 2c:")
trace_shift(xs2, 3, 5)
print("Bài 2d:")
_ = trace_two_pointers(xs2)          # gán vào _ để Jupyter không in thêm giá trị trả về

Bài 1a: [0, 1, 1, 2]
Bài 1b: [1, 0, 2, 1]
Bài 1c: [0, 1, 1, 2]
Bài 1d: 4 | ô tối ưu: (1, 1) | có nhà? False

Bài 2a: mean = 6.0 -> cost(6) = 18
Bài 2b: median = 4 -> cost(4) = 16
Bài 2c:
 x | L(<=x) R(>x) | cost(x) | cost(x+1)-cost(x) = L-R
 3 |   1      4   |     19  |  -3  sang phải có lợi
 4 |   3      2   |     16  |  +1  sang phải bị thiệt
 5 |   3      2   |     17  |  +1  sang phải bị thiệt
Bài 2d:
  points = [1, 4, 4, 6, 15]
    i=0, j=4: cặp (1, 15) -> +14, total = 14
    i=1, j=3: cặp (4, 6) -> +2, total = 16
    dừng vì i=2 >= j=2; kết quả 16


In [36]:
# ---------- Bài 3 ----------
def answer_min_distance_1d(points):
    i, j = 0, len(points) - 1
    total = 0
    while i < j:
        total += points[j] - points[i]
        i += 1
        j -= 1
    return total

# ---------- Bài 4 ----------
def answer_min_total_distance(grid):
    m, n = len(grid), len(grid[0])
    rows = [r for r in range(m) for c in range(n) if grid[r][c] == 1]   # duyệt theo hàng
    cols = [c for c in range(n) for r in range(m) if grid[r][c] == 1]   # duyệt theo cột
    return answer_min_distance_1d(rows) + answer_min_distance_1d(cols)

# ---------- Bài 5 (LeetCode 462) ----------
def answer_min_moves2(nums):
    return answer_min_distance_1d(sorted(nums))

# ---------- Bài 6 (mỗi ô nhiều người) ----------
def answer_weighted_1d(count):
    """count[i] = số người ở vị trí i. Trả về min_x Σ count[i] * |i - x|."""
    people = sum(count)
    prefix, median = 0, 0
    for i, cnt in enumerate(count):
        prefix += cnt
        if 2 * prefix >= people:          # lần đầu bên trái-hoặc-tại chiếm >= một nửa
            median = i
            break
    return sum(cnt * abs(i - median) for i, cnt in enumerate(count))

def answer_min_total_weighted(grid):
    m, n = len(grid), len(grid[0])
    row_count, col_count = [0] * m, [0] * n           # bộ nhớ phụ O(m + n)
    for r in range(m):
        for c in range(n):
            row_count[r] += grid[r][c]
            col_count[c] += grid[r][c]
    return answer_weighted_1d(row_count) + answer_weighted_1d(col_count)

# ---------- Kiểm tra ----------
assert answer_min_distance_1d([1, 2, 7, 9, 10]) == 16
assert answer_min_distance_1d([1, 2, 7, 9]) == 13
assert answer_min_distance_1d([5, 5]) == 0
assert answer_min_distance_1d([0, 3]) == 3

assert answer_min_total_distance([[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]]) == 6
assert answer_min_total_distance([[1, 1]]) == 1
assert answer_min_total_distance([[1], [1], [1]]) == 2
assert answer_min_total_distance(grid_trace) == 16

assert answer_min_moves2([1, 2, 3]) == 2
assert answer_min_moves2([1, 10, 2, 9]) == 16
assert answer_min_moves2([7]) == 0
assert answer_min_moves2([-5, 5]) == 10

assert answer_min_total_weighted([[2, 0, 1]]) == 2
assert answer_min_total_weighted([[1, 0, 0], [0, 0, 0], [0, 0, 3]]) == 4
assert answer_min_total_weighted([[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]]) == 6
assert answer_min_total_weighted([[0, 1000000, 0, 1]]) == 2

# Đối chiếu ngẫu nhiên: Bài 4 và Bài 6 (trên lưới 0/1) khớp Solution;
# Bài 6 trên lưới nhiều người khớp brute force (nhân bản mỗi người thành một "nhà")
random.seed(462)
for _ in range(300):
    m, n = random.randint(1, 5), random.randint(1, 5)
    if m * n < 2:
        continue
    g01 = random_grid(m, n, random.random())
    assert answer_min_total_distance(g01) == answer_min_total_weighted(g01) == Solution().minTotalDistance(g01)
    gw = [[random.choice([0, 0, 1, 2, 3]) for _ in range(n)] for _ in range(m)]
    gw[0][0] += 1                                       # bảo đảm có ít nhất 1 người
    expanded = [(r, c) for r in range(m) for c in range(n) for _ in range(gw[r][c])]
    brute = min(total_distance(expanded, R, C) for R in range(m) for C in range(n))
    assert answer_min_total_weighted(gw) == brute, (gw, brute)
    nums = [random.randint(-20, 20) for _ in range(random.randint(1, 8))]
    assert answer_min_moves2(nums) == min(cost_1d(nums, x) for x in range(-20, 21))
print("Đáp án Bài 3–6: PASS (kể cả đối chiếu ngẫu nhiên)")

Đáp án Bài 3–6: PASS (kể cả đối chiếu ngẫu nhiên)


## 20. Template ngắn nên nhớ

```python
class Solution:
    def minTotalDistance(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        rows = [r for r in range(m) for c in range(n) if grid[r][c] == 1]   # theo hàng -> tăng dần
        cols = [c for c in range(n) for r in range(m) if grid[r][c] == 1]   # theo cột  -> tăng dần

        def min_1d(p):                     # p đã sắp xếp
            i, j, total = 0, len(p) - 1, 0
            while i < j:
                total += p[j] - p[i]       # cặp xa nhau nhất
                i, j = i + 1, j - 1
            return total

        return min_1d(rows) + min_1d(cols)
```

Câu thần chú: **"Manhattan thì tách trục — mỗi trục lấy median — duyệt đúng chiều khỏi phải sort — hai con trỏ cộng cặp xa nhất."**

## Checklist trước khi submit

- [ ] Dùng công thức `|dr| + |dc|` thay vì BFS (lưới **không có vật cản**)?
- [ ] Tối ưu **riêng** phần hàng và phần cột rồi cộng lại?
- [ ] Dùng **median** (hoặc hai con trỏ), **không** dùng trung bình cộng?
- [ ] List đưa vào median / hai con trỏ **đã tăng dần**: `rows` gom theo hàng, `cols` gom theo **cột** (hoặc đã sort)?
- [ ] Hai con trỏ dùng `while i < j` (không phải `i != j`)?
- [ ] `m = len(grid)` (số hàng) và `n = len(grid[0])` (số cột) đặt đúng chỗ khi đảo vòng lặp?
- [ ] Trả về **tổng quãng đường** (số nguyên), không phải tọa độ điểm hẹn?
- [ ] Đã test: 2 example của đề, lưới 1 hàng, lưới 1 cột, số người chẵn, điểm hẹn tối ưu rơi vào ô trống?